# Causal occlusion audit of brain-tumour MRI classifiers on BRISC 2025

This notebook reproduces every number, table and figure of the accompanying paper. It audits whether
convolutional classifiers trained on the BRISC 2025 four-class MRI dataset base their predictions on the tumour,
using an area-matched, three-arm causal occlusion audit with shape-matched null distributions, validated on
synthetic shortcuts with known ground truth.

## What the notebook does

| Step | Content |
|---|---|
| Dataset audit | source properties per class, metadata-only classifier, pixel-confirmed duplicates (test↔train/val and within test) |
| Fingerprint probes | linear probes on frozen ImageNet features of tumour-free patches, noise residuals and background-only images |
| Synthetic validation | three shortcut conditions (corner text 100/0, corner text 80/20, arrow inside the head) × 3 seeds, with an oracle arm as ground truth and clean models as negative controls |
| Model audit | ResNet50, DenseNet121, EfficientNet-B0 × 3 seeds; five explanation maps (spectral, spectral subspace, CAM, Eigen-CAM, Integrated Gradients) |
| Robustness | two occlusion budgets (1×, 2× tumour area), three fill methods, all vs deduplicated test set |

## Fixed analysis choices

- Primary analysis: clean models, `nconv` fill, budget **2×**, **deduplicated** test set. Budget 1×, the full
  test set and the other fills are reported as sensitivity analyses.
- Verdict thresholds: q < 0.05 (Benjamini–Hochberg), rank-biserial r ≥ 0.30, effect ≥ 5% of the median baseline margin.
- Synthetic validation: ground truth = oracle arm. A map arm counts as a detection when the verdict flags an
  off-tumour region and ≥ 50% of that region lies within ±32 px of the marker.

## Running on Kaggle

1. **Accelerator: GPU T4** (one GPU is used), **Settings → Internet: On** (ImageNet weights).
2. Add the BRISC 2025 dataset as input; the path is detected automatically.
3. Optional: run once with `QUICK_TEST = True` (~15–25 min) to check the pipeline.
4. Set `QUICK_TEST = False` and use **Save Version → Save & Run All (Commit)**. A full run takes ~4.5 h on a T4.
   A safety limit (`MAX_HOURS = 11`) skips remaining runs if needed; aggregation still runs.
5. Outputs: folder `brisc_audit/` (CSV tables, PDF/PNG figures) and `brisc_audit_results.zip`.

The notebook resumes automatically: finished checkpoints, spectral fits and audits are skipped.
If a run fails, the error is written to `errors.log` and the next run continues.

In [ ]:
# ===================================================================
# 0) CONFIGURATION — every experimental parameter is set in this cell
# ===================================================================
import os

QUICK_TEST = False          # True → fast pipeline check on a small subset; do NOT report these results

BASE_DIR   = '/kaggle/input/datasets/briscdataset/brisc2025/brisc2025'   # auto-detected if different
OUT_DIR    = '/kaggle/working/brisc_audit'
PRETRAINED = True           # ImageNet weights (requires Internet: On)

CLASS_NAMES = ['no_tumor', 'glioma', 'meningioma', 'pituitary']   # labels 0–3
IMG_SIZE    = 224
SPLIT_SEED  = 42            # fixed train/val split
VAL_FRAC    = 0.20

# --- Clean models ---
ARCHS = ['resnet50', 'densenet121', 'efficientnet_b0']
SEEDS = [0, 1, 2]

# --- Training (Adam 1e-4, StepLR 4/0.1, 10 epochs, batch 32) ---
EPOCHS, BATCH_SIZE, LR, STEP_SIZE, GAMMA = 10, 32, 1e-4, 4, 0.1

# --- Spectral analysis (fitted on training features) ---
PA_PERMUTATIONS   = 19
PA_COMPONENTS     = 50
N_RANDOM_SUBSPACE = 20

# --- Three-arm audit ---
MAPS           = ['spectral', 'spectral_subspace', 'cam', 'eigencam', 'ig']
BUDGETS        = [1.0, 2.0]   # occluded area = b × tumour area (no_tumor: b × median training tumour area)
PRIMARY_BUDGET = 2.0          # budget of the primary analysis (fixed before the confirmatory run)
N_RAND         = 30
FILL_MODE      = 'nconv'      # 'nconv' | 'blur' | 'black' | 'imagenet_mean'
SENSITIVITY_FILLS = ['blur', 'imagenet_mean']   # repeated on the first run
MAX_TUMOR_OVERLAP = 0.10
MAX_SELF_OVERLAP  = 0.10
FG_TOL            = 0.15
MIN_VALID_TRIALS  = 10
PERI_RING_PX      = 15
BORDER_PX         = 16
NCONV_SIGMA       = 24.0
AUDIT_BATCH       = 16
FORWARD_CHUNK     = 256
IG_STEPS          = 20        # Integrated Gradients: number of steps (midpoint rule)
IG_SMOOTH_SIGMA   = 4.0       # Gaussian smoothing of IG maps (pixels) so top-k regions are compact

# --- Statistics & verdict rules ---
PRIMARY_METRIC = 'margin'
ONLY_CORRECT   = True
ALPHA          = 0.05
R_MIN          = 0.30
REL_MIN        = 0.05
IN_TUMOR_MAX   = 0.25
OOD_FLIP       = 0.10
INERT_FLIP     = 0.02
N_BOOT         = 2000
N_NULL         = 2000

# --- Synthetic shortcut validation (ground truth) ---
RUN_SYNTHETIC = True
SYN_ARCH      = 'resnet50'
SYN_SEEDS     = [0, 1, 2]
SYN_TARGET    = 'meningioma'
SYN_CONDITIONS = {   # condition names without underscores (used in folder names)
    'cornerR':    dict(kind='corner_text', p_target=1.0, p_other=0.0, seed=123),   # full shortcut
    'cornerRp80': dict(kind='corner_text', p_target=0.8, p_other=0.2, seed=133),   # partial shortcut
    'arrowHead':  dict(kind='head_arrow',  p_target=1.0, p_other=0.0, seed=143),   # artefact inside the anatomy
}
SYN_TEXT       = 'R'
ARROW_LEN      = 22           # arrow length (pixels at 224 px)
MARKER_ZONE_PX = 32
SYN_AUDIT_TARGET_ONLY = True  # synthetic audits on target-class test images only (3× faster)

# --- Dataset audit ---
DUP_HAMMING_CAND = 4          # duplicate candidates: dHash Hamming distance ≤ 4
DUP_PIXEL_DIFF   = 3.0        # confirmed: mean |pixel difference| < 3 (0–255 scale, 224 px)

# --- Fingerprint probes (linear probes on frozen ImageNet features) ---
RUN_FINGERPRINT = True
FP_PATCH        = 56          # patch size (pixels at 224 px)
FP_PER_IMAGE    = 4
FP_MIN_FG       = 0.95        # patch must lie ≥ 95% inside the head
FP_EXCL_DIL     = 24          # patch must be ≥ 24 px away from the tumour
FP_INPUT        = 112
FP_PCA          = 256
FP_RESID_SIGMA  = 1.5

# --- Other ---
MAX_HOURS       = 11.0
FORCE_RERUN     = False
QUICK_TRAIN_PER_CLASS, QUICK_TEST_PER_CLASS = 60, 25

if QUICK_TEST:
    OUT_DIR = OUT_DIR.rstrip('/') + '_quick'
    ARCHS, SEEDS, EPOCHS = ARCHS[:1], [0], 2
    SYN_SEEDS = [0]
    N_RAND, PA_PERMUTATIONS, N_RANDOM_SUBSPACE = 8, 3, 3
    N_BOOT, N_NULL = 300, 300
    SENSITIVITY_FILLS = SENSITIVITY_FILLS[:1]
    IG_STEPS, FP_PER_IMAGE = 8, 2

In [ ]:
# ===================================================================
# 1) IMPORTS, SEEDS, PATH DETECTION
# ===================================================================
import json, math, time, random, re, glob, zipfile, hashlib, warnings, traceback, contextlib
from concurrent.futures import ThreadPoolExecutor
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')   # must be set before CUDA is initialised

import numpy as np
import pandas as pd
import cv2
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
from torchvision import models
from scipy import stats, ndimage
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import balanced_accuracy_score
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
T_GLOBAL = time.time()
DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
torch.use_deterministic_algorithms(True, warn_only=True)

N_CLASSES  = len(CLASS_NAMES)
TARGET_IDX = CLASS_NAMES.index(SYN_TARGET)
PRIMARY    = 'dM' if PRIMARY_METRIC == 'margin' else 'dL'
MIN_VALID_TRIALS = min(MIN_VALID_TRIALS, max(2, N_RAND // 2))
SYN_CONDS  = list(SYN_CONDITIONS) if RUN_SYNTHETIC else []
plt.rcParams.update({'figure.dpi': 100, 'savefig.dpi': 300, 'font.size': 8,
                     'axes.titlesize': 9, 'axes.spines.top': False, 'axes.spines.right': False})


def hours_used():
    return (time.time() - T_GLOBAL) / 3600.0


def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def amp_ctx():
    return torch.autocast(device_type='cuda', dtype=torch.float16) if USE_AMP else contextlib.nullcontext()


def find_brisc_root(preferred):
    if os.path.isdir(os.path.join(preferred, 'classification_task')):
        return preferred
    for top in ['/kaggle/input', os.path.dirname(preferred.rstrip('/'))]:
        if not os.path.isdir(top):
            continue
        for root, dirs, _ in os.walk(top):
            if root[len(top):].count(os.sep) > 6:
                dirs[:] = []
                continue
            if 'classification_task' in dirs:
                return root
    raise FileNotFoundError('BRISC 2025 folder (containing classification_task/) not found. '
                            'Add the BRISC 2025 dataset to the notebook or set BASE_DIR.')


def _to_py(o):
    if isinstance(o, np.integer): return int(o)
    if isinstance(o, np.floating): return float(o)
    if isinstance(o, np.bool_): return bool(o)
    if isinstance(o, np.ndarray): return o.tolist()
    raise TypeError(type(o))


def save_json(obj, path):
    with open(path, 'w') as f:
        json.dump(obj, f, indent=2, default=_to_py)


def load_json(path):
    with open(path) as f:
        return json.load(f)


def save_fig(fig, path, show=False):
    fig.savefig(path, bbox_inches='tight')
    if path.endswith('.pdf'):
        fig.savefig(path[:-4] + '.png', bbox_inches='tight', dpi=150)
    if show:
        plt.show()
    plt.close(fig)


def to_bool(s):
    """Boolean column that may have been read back from CSV as text."""
    return s.astype(str).str.strip().str.lower().isin(['true', '1', '1.0'])


BASE_DIR = find_brisc_root(BASE_DIR)
for sub in ['cache', 'runs', 'figures', 'tables']:
    os.makedirs(os.path.join(OUT_DIR, sub), exist_ok=True)
TD, FD = os.path.join(OUT_DIR, 'tables'), os.path.join(OUT_DIR, 'figures')

print(f'torch {torch.__version__} | torchvision {torchvision.__version__} | device: {DEVICE}'
      + (f' ({torch.cuda.get_device_name(0)})' if DEVICE.type == 'cuda' else ''))
print('BRISC root :', BASE_DIR)
print('Output     :', OUT_DIR)
print('QUICK_TEST :', QUICK_TEST)
if DEVICE.type != 'cuda':
    print('WARNING: no GPU detected. Enable a GPU accelerator (Settings → Accelerator).')
save_json({k: v for k, v in list(globals().items())
           if k.isupper() and isinstance(v, (int, float, str, bool, list, dict))},
          os.path.join(OUT_DIR, 'config.json'))

## 2. Data: 224 px cache, tumour masks, head foreground, original file properties

Besides the 224 px images, this cell reads the **original size and file mode** (L/RGB) from each file header and
computes background statistics (outside the head) on the 224 px images seen by the models. These properties are
used in the dataset audit.

In [ ]:
# ===================================================================
# 2) DATA
# ===================================================================
IMG_EXT = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')
PLANE_TOKENS = {'ax': 'axial', 'axial': 'axial', 'tra': 'axial', 'co': 'coronal', 'cor': 'coronal',
                'coronal': 'coronal', 'sa': 'sagittal', 'sag': 'sagittal', 'sagittal': 'sagittal'}
POP8 = np.array([bin(i).count('1') for i in range(256)], dtype=np.uint8)
DIL11 = np.ones((11, 11), np.uint8)       # 5 px dilation (same as the background arm)


def parse_plane(stem):
    for tok in re.split(r'[_\-.\s]+', stem.lower()):
        if tok in PLANE_TOKENS:
            return PLANE_TOKENS[tok]
    return 'unknown'


def index_masks(base):
    idx = {}
    for root, _, files in os.walk(os.path.join(base, 'segmentation_task')):
        if 'mask' not in os.path.basename(root).lower():
            continue
        for f in files:
            if f.lower().endswith(IMG_EXT):
                idx.setdefault(os.path.splitext(f)[0], []).append(os.path.join(root, f))
    return idx


def build_records(base):
    mask_idx = index_masks(base)
    recs = []
    for split in ['train', 'test']:
        for label, cls in enumerate(CLASS_NAMES):
            d = os.path.join(base, 'classification_task', split, cls)
            if not os.path.isdir(d):
                print('WARNING: folder not found:', d)
                continue
            for f in sorted(os.listdir(d)):
                if not f.lower().endswith(IMG_EXT):
                    continue
                stem = os.path.splitext(f)[0]
                mpath = None
                if label != 0:
                    cands = mask_idx.get(stem, [])
                    same = [c for c in cands if f'{os.sep}{split}{os.sep}' in c]
                    mpath = (same or cands or [None])[0]
                recs.append(dict(split=split, name=f, stem=stem, label=label, cls=cls,
                                 path=os.path.join(d, f), mask_path=mpath, plane=parse_plane(stem)))
    return pd.DataFrame(recs)


def head_mask(img):
    """Head foreground: blur → threshold → largest component → fill holes."""
    g = cv2.GaussianBlur(img, (0, 0), 2)
    thr = max(10.0, 0.10 * float(np.percentile(g, 99)))
    fg = (g > thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(fg, connectivity=8)
    if n > 1:
        fg = (lab == 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))).astype(np.uint8)
    return ndimage.binary_fill_holes(fg > 0)


def dhash64(img):
    s = cv2.resize(img, (9, 8), interpolation=cv2.INTER_AREA).astype(np.int16)
    return np.packbits((s[:, 1:] > s[:, :-1]).ravel()).view('>u8')[0]


def load_one(rec):
    try:
        with Image.open(rec['path']) as im:
            w0, h0, mode0 = int(im.size[0]), int(im.size[1]), str(im.mode)
    except Exception:
        raw = cv2.imread(rec['path'], cv2.IMREAD_UNCHANGED)
        h0, w0 = (raw.shape[:2] if raw is not None else (0, 0))
        mode0 = 'unknown' if raw is None else ('L' if raw.ndim == 2 else {3: 'RGB', 4: 'RGBA'}.get(raw.shape[2], 'other'))
    img = cv2.imread(rec['path'], cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise IOError(f'Cannot read {rec["path"]}')
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    m = np.zeros((IMG_SIZE, IMG_SIZE), bool)
    mpath = rec['mask_path']
    if isinstance(mpath, str) and mpath:          # None/NaN = no mask
        mm = cv2.imread(mpath, cv2.IMREAD_GRAYSCALE)
        if mm is not None and mm.max() > 0:      # works for 0/1 and 0/255 masks
            mm = cv2.resize(mm.astype(np.float32) / float(mm.max()), (IMG_SIZE, IMG_SIZE),
                            interpolation=cv2.INTER_LINEAR)
            m = mm > 0.5
    return img, m, head_mask(img), dhash64(img), w0, h0, mode0


def bg_stats(img, fg):
    """Background statistics (outside the 5-px-dilated head) on the 224 px image."""
    bg = ~(cv2.dilate(fg.astype(np.uint8), DIL11) > 0)
    v = img[bg].astype(np.float32)
    if v.size == 0:
        return np.nan, np.nan, np.nan, 0
    head_p99 = float(np.percentile(img[fg], 99)) if fg.any() else 255.0
    bright = (img > max(25.0, 0.25 * head_p99)) & bg
    n_lab, _, st, _ = cv2.connectedComponentsWithStats(bright.astype(np.uint8), connectivity=8)
    marks = int((st[1:, cv2.CC_STAT_AREA] >= 15).sum()) if n_lab > 1 else 0
    return float(v.mean()), float(v.std()), float((v > 5).mean()), marks


def load_dataset():
    recs = build_records(BASE_DIR)
    if len(recs) == 0:
        raise RuntimeError('No images found in classification_task/.')
    if QUICK_TEST:
        parts = []
        for (sp, _), g in recs.groupby(['split', 'label']):
            n = QUICK_TRAIN_PER_CLASS if sp == 'train' else QUICK_TEST_PER_CLASS
            parts.append(g.sample(n=min(n, len(g)), random_state=0))
        recs = pd.concat(parts).sort_values(['split', 'label', 'name']).reset_index(drop=True)
    key = hashlib.md5(('|'.join(recs.path) + f'|{IMG_SIZE}|cache1').encode()).hexdigest()[:10]
    cache = os.path.join(OUT_DIR, 'cache', f'brisc_{IMG_SIZE}_{key}.npz')
    if os.path.exists(cache):
        z = np.load(cache)
        imgs, masks, fg, dh = z['imgs'], z['masks'], z['fg'], z['dh']
        ow, oh, om = z['ow'], z['oh'], z['om']
        print('Cache loaded:', cache)
    else:
        with ThreadPoolExecutor(max_workers=8) as ex:
            res = list(tqdm(ex.map(load_one, recs.to_dict('records')), total=len(recs), desc='Reading images'))
        imgs = np.stack([r[0] for r in res])
        masks = np.stack([r[1] for r in res])
        fg = np.stack([r[2] for r in res])
        dh = np.array([r[3] for r in res], dtype=np.uint64)
        ow = np.array([r[4] for r in res], dtype=np.int32)
        oh = np.array([r[5] for r in res], dtype=np.int32)
        om = np.array([r[6] for r in res], dtype='<U8')
        np.savez(cache, imgs=imgs, masks=masks, fg=fg, dh=dh, ow=ow, oh=oh, om=om)
    recs['tumor_area'] = masks.reshape(len(recs), -1).sum(1)
    recs['mask_ok'] = (recs.label.values != 0) & (recs.tumor_area.values >= 10)
    recs['orig_w'], recs['orig_h'], recs['file_mode'] = ow, oh, om
    st = [bg_stats(imgs[i], fg[i]) for i in range(len(recs))]
    recs['bg_mean'] = [s[0] for s in st]
    recs['bg_std'] = [s[1] for s in st]
    recs['bg_nonzero'] = [s[2] for s in st]
    recs['bg_marks'] = [s[3] for s in st]
    return recs, imgs, masks, fg, dh


META, IMGS, MASKS, FG, DHASH = load_dataset()

# Split: train/val from the official train folder (stratified, fixed seed); official test set held out
_tr_all = np.flatnonzero(META.split.values == 'train')
IDX_TR, IDX_VA = train_test_split(_tr_all, test_size=VAL_FRAC, random_state=SPLIT_SEED,
                                  stratify=META.label.values[_tr_all])
IDX_TR, IDX_VA = np.sort(IDX_TR), np.sort(IDX_VA)
IDX_TE = np.flatnonzero(META.split.values == 'test')
META['subset'] = 'test'
META.loc[IDX_TR, 'subset'] = 'train'
META.loc[IDX_VA, 'subset'] = 'val'
save_json({s: META.path.values[ix].tolist() for s, ix in [('train', IDX_TR), ('val', IDX_VA), ('test', IDX_TE)]},
          os.path.join(OUT_DIR, 'split.json'))

_ok_tr = IDX_TR[META.mask_ok.values[IDX_TR]]
REF_AREA = float(np.median(META.tumor_area.values[_ok_tr])) if len(_ok_tr) else 0.03 * IMG_SIZE ** 2

print('\nImages per subset × class:')
print(pd.crosstab(META.subset, META.cls).reindex(index=['train', 'val', 'test'], columns=CLASS_NAMES))
_cov = META[META.label > 0].groupby(['subset', 'cls']).mask_ok.agg(['sum', 'count'])
_cov.columns = ['with_mask', 'total']
print('\nTumour mask availability (images without a mask are excluded from Arm 3):')
print(_cov)
print('\nTumour area (% of image), images with a mask:')
print((META[META.mask_ok].groupby('cls').tumor_area.describe()[['count', '25%', '50%', '75%']]
       / np.array([1, IMG_SIZE ** 2 / 100, IMG_SIZE ** 2 / 100, IMG_SIZE ** 2 / 100])).round(2))
print(f'REF_AREA (median training tumour area) = {REF_AREA:.0f} px = {100 * REF_AREA / IMG_SIZE ** 2:.2f}% of the image')
if (META.plane != 'unknown').mean() > 0.5:
    print('\nImaging plane per class:')
    print(pd.crosstab(META.cls, META.plane).reindex(CLASS_NAMES))


# Expanded tumour region (Arm 3 for budgets > 1×): tumour + nearest pixels until the area is exactly b×
def expand_to_area(mask, target):
    area = int(mask.sum())
    target = int(min(target, IMG_SIZE * IMG_SIZE // 2))
    if area == 0 or target <= area:
        return mask.copy()
    dist = cv2.distanceTransform((~mask).astype(np.uint8), cv2.DIST_L2, 5)
    cand = np.flatnonzero(~mask.ravel())
    d = dist.ravel()[cand] + np.random.default_rng(0).random(len(cand)) * 1e-3
    out = mask.copy()
    out.reshape(-1)[cand[np.argsort(d, kind='stable')[:target - area]]] = True
    return out


POS_TE = np.full(len(META), -1)
POS_TE[IDX_TE] = np.arange(len(IDX_TE))
TUMOR_EXP = {}
for _b in BUDGETS:
    if _b == 1.0:
        continue
    _arr = np.zeros((len(IDX_TE), IMG_SIZE, IMG_SIZE), bool)
    for _p, _g in enumerate(IDX_TE):
        if META.mask_ok.values[_g]:
            _arr[_p] = expand_to_area(MASKS[_g], round(_b * META.tumor_area.values[_g]))
    TUMOR_EXP[_b] = _arr


def tumor_region(gidx, b):
    """Arm-3 region for global indices gidx (test images only)."""
    if b == 1.0:
        return MASKS[gidx] & META.mask_ok.values[gidx][:, None, None]
    return TUMOR_EXP[b][POS_TE[gidx]]


_fig, _ax = plt.subplots(2, N_CLASSES, figsize=(2.3 * N_CLASSES, 4.6))
for _c in range(N_CLASSES):
    _cand = IDX_TE[META.label.values[IDX_TE] == _c]
    if len(_cand) == 0:
        continue
    _g = _cand[0]
    _ax[0, _c].imshow(IMGS[_g], cmap='gray'); _ax[0, _c].set_title(CLASS_NAMES[_c])
    _ax[1, _c].imshow(IMGS[_g], cmap='gray')
    _ax[1, _c].contour(FG[_g], levels=[0.5], colors='y', linewidths=0.7)
    if MASKS[_g].any():
        _ax[1, _c].contour(MASKS[_g], levels=[0.5], colors='lime', linewidths=0.8)
    for _a in _ax[:, _c]:
        _a.axis('off')
_ax[1, 0].set_title('yellow: head foreground, green: tumour', fontsize=7)
save_fig(_fig, os.path.join(FD, 'data_examples.pdf'), show=True)

## 2b. Dataset audit: image sources, metadata classifier, duplicates

- **Source properties per class**: original size, file mode and non-black background. None of these carry
  pathology information, so any association with the class label is a confound.
- **Metadata classifier**: a random forest that sees only original size, file mode and background statistics.
  Its accuracy measures how well the class can be guessed without looking at the brain.
- **Duplicates**: dHash candidates (Hamming ≤ 4), confirmed when the mean absolute pixel difference is < 3 at
  224 px. Searched test↔train/val, val↔train and within test. The `dedup` subset removes test images with a copy in
  train/val and keeps a single member of each within-test duplicate cluster.

In [ ]:
# ===================================================================
# 2b) DATASET AUDIT
# ===================================================================
META['is_rgb'] = (META.file_mode.values == 'RGB').astype(int)
_sq512 = (META.orig_w.values == 512) & (META.orig_h.values == 512)
META['size_group'] = np.where(_sq512, '512x512', np.where(META.orig_w.values == META.orig_h.values,
                                                           'square, other size', 'non-square'))


def candidate_pairs(q_idx, r_idx, max_h, exclude_self=False, max_per_query=50, chunk=256):
    """(query, reference, Hamming) pairs with Hamming ≤ max_h; at most max_per_query nearest per query."""
    out = []
    Rh = DHASH[r_idx]
    for s in range(0, len(q_idx), chunk):
        qi = q_idx[s:s + chunk]
        x = np.bitwise_xor(DHASH[qi][:, None], Rh[None, :])
        d = POP8[x.view(np.uint8)].reshape(x.shape[0], x.shape[1], 8).sum(-1)
        for a in range(len(qi)):
            js = np.flatnonzero(d[a] <= max_h)
            if exclude_self:
                js = js[r_idx[js] > qi[a]]
            if len(js) > max_per_query:
                js = js[np.argsort(d[a, js], kind='stable')[:max_per_query]]
            out += [(int(qi[a]), int(r_idx[j]), int(d[a, j])) for j in js]
    return out


def pixdiff(a, b):
    return float(np.abs(IMGS[a].astype(np.int16) - IMGS[b].astype(np.int16)).mean())


_trv = np.r_[IDX_TR, IDX_VA]
_pairs = []
for _kind, _q, _r, _ex in [('test-trainval', IDX_TE, _trv, False), ('test-test', IDX_TE, IDX_TE, True),
                           ('val-train', IDX_VA, IDX_TR, False)]:
    for _a, _b2, _h in candidate_pairs(_q, _r, DUP_HAMMING_CAND, exclude_self=_ex):
        _pairs.append(dict(pair_type=_kind, a=_a, b=_b2, hamming=_h, pixel_diff=pixdiff(_a, _b2)))
DUPP = pd.DataFrame(_pairs) if _pairs else pd.DataFrame(
    {'pair_type': pd.Series(dtype=object), 'a': pd.Series(dtype=int), 'b': pd.Series(dtype=int),
     'hamming': pd.Series(dtype=int), 'pixel_diff': pd.Series(dtype=float)})
DUPP['confirmed'] = (DUPP.pixel_diff.astype(float) < DUP_PIXEL_DIFF).astype(bool)
DUPP['name_a'] = META.name.values[DUPP.a.values.astype(int)] if len(DUPP) else []
DUPP['name_b'] = META.name.values[DUPP.b.values.astype(int)] if len(DUPP) else []
DUPP['cls_a'] = META.cls.values[DUPP.a.values.astype(int)] if len(DUPP) else []
DUPP['cls_b'] = META.cls.values[DUPP.b.values.astype(int)] if len(DUPP) else []
DUPP.sort_values(['pair_type', 'pixel_diff']).to_csv(os.path.join(TD, 'T0c_duplicate_pairs.csv'), index=False)

_dup_train = set(DUPP[(DUPP.pair_type == 'test-trainval') & DUPP.confirmed].a.astype(int))
_parent = {}


def _find(a):
    _parent.setdefault(a, a)
    while _parent[a] != a:
        _parent[a] = _parent[_parent[a]]
        a = _parent[a]
    return a


for _a, _b2 in DUPP[(DUPP.pair_type == 'test-test') & DUPP.confirmed][['a', 'b']].astype(int).itertuples(index=False):
    _ra, _rb = _find(_a), _find(_b2)
    if _ra != _rb:
        _parent[max(_ra, _rb)] = min(_ra, _rb)
_clusters = {}
for _a in list(_parent):
    _clusters.setdefault(_find(_a), []).append(_a)
_within_drop = set(m for mem in _clusters.values() if len(mem) > 1 for m in sorted(mem)[1:])
DUP_TEST = _dup_train | _within_drop
META['dup'] = np.isin(np.arange(len(META)), sorted(DUP_TEST))

_te = META.loc[IDX_TE]
T0d = pd.DataFrame({
    'n_test': _te.groupby('cls').size(),
    'dup_of_train_or_val': _te[_te.index.isin(sorted(_dup_train))].groupby('cls').size(),
    'within_test_extra_copies': _te[_te.index.isin(sorted(_within_drop - _dup_train))].groupby('cls').size(),
}).reindex(CLASS_NAMES).fillna(0).astype(int)
T0d['dedup_remaining'] = T0d.n_test - _te.groupby('cls').dup.sum().reindex(CLASS_NAMES).fillna(0).astype(int)
T0d['pct_removed'] = (100 * (1 - T0d.dedup_remaining / T0d.n_test.clip(lower=1))).round(1)
T0d.to_csv(os.path.join(TD, 'T0d_duplicates_by_class.csv'))
_conf = DUPP[DUPP.confirmed]
_conflict = _conf[_conf.cls_a != _conf.cls_b]
print('Confirmed duplicates per class (test):')
print(T0d)
print(f'\nConfirmed pairs: test↔train/val = {int((_conf.pair_type == "test-trainval").sum())}, '
      f'within test = {int((_conf.pair_type == "test-test").sum())} ({len([c for c in _clusters.values() if len(c) > 1])} clusters), '
      f'val↔train = {int((_conf.pair_type == "val-train").sum())}')
print(f'dHash candidates NOT confirmed at pixel level: {int((~DUPP.confirmed).sum())} pairs')
print(f'Confirmed pairs with conflicting labels: {len(_conflict)}')
if len(_conflict):
    print(_conflict[['pair_type', 'name_a', 'cls_a', 'name_b', 'cls_b', 'pixel_diff']].head(20).to_string(index=False))
print(f'Deduplicated test subset: {len(IDX_TE) - len(DUP_TEST)} of {len(IDX_TE)} test images')

# Source properties per class
T0a = META.groupby('cls').agg(
    n=('name', 'size'),
    pct_512x512=('size_group', lambda v: 100 * (v == '512x512').mean()),
    pct_non_square=('size_group', lambda v: 100 * (v == 'non-square').mean()),
    pct_rgb_file=('is_rgb', lambda v: 100 * v.mean()),
    pct_bg_nonblack=('bg_nonzero', lambda v: 100 * (v > 0.2).mean()),
    bg_mean_median=('bg_mean', 'median'),
    pct_bright_marks_outside_head=('bg_marks', lambda v: 100 * (v > 0).mean())).reindex(CLASS_NAMES).round(2)
T0a.to_csv(os.path.join(TD, 'T0a_source_properties.csv'))
META[['name', 'split', 'subset', 'cls', 'plane', 'orig_w', 'orig_h', 'file_mode', 'size_group', 'bg_mean', 'bg_std',
      'bg_nonzero', 'bg_marks', 'tumor_area', 'mask_ok', 'dup']].to_csv(
    os.path.join(TD, 'per_image_properties.csv'), index_label='gidx')
print('\nSource properties per class (all splits):')
print(T0a.to_string())
_sz = (META.orig_w.astype(str) + 'x' + META.orig_h.astype(str))
_top = _sz.value_counts().index[:6]
print('\nOriginal image size (fraction per class, 6 most frequent sizes):')
print(pd.crosstab(META.cls, _sz.where(_sz.isin(_top), 'other'), normalize='index').reindex(CLASS_NAMES).round(3))

# Metadata classifier (train → test)
MD_FEATS = {'size + file mode': ['orig_w', 'orig_h', 'is_rgb'],
            'background statistics': ['bg_mean', 'bg_std', 'bg_nonzero', 'bg_marks'],
            'all metadata': ['orig_w', 'orig_h', 'is_rgb', 'bg_mean', 'bg_std', 'bg_nonzero', 'bg_marks']}


def score_predictions(pred, g, name, subset):
    g, pred = np.asarray(g), np.asarray(pred)
    keep = np.ones(len(g), bool) if subset == 'all' else ~META.dup.values[g]
    g, p = g[keep], pred[keep]
    y = META.label.values[g]
    rec = {f'recall_{c}': float((p[y == i] == i).mean()) if (y == i).any() else np.nan
           for i, c in enumerate(CLASS_NAMES)}
    return dict(probe=name, subset=subset, n_images=len(g), acc=float((p == y).mean()) if len(g) else np.nan,
                bal_acc=float(balanced_accuracy_score(y, p)) if len(g) else np.nan, **rec)


PROBE_ROWS = []
for _name, _cols in MD_FEATS.items():
    _rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=3, random_state=0, n_jobs=-1)
    _rf.fit(META.loc[IDX_TR, _cols].fillna(0).values, META.label.values[IDX_TR])
    _pred = _rf.predict(META.loc[IDX_TE, _cols].fillna(0).values)
    for _sub in ['dedup', 'all']:
        PROBE_ROWS.append(score_predictions(_pred, IDX_TE, f'metadata RF: {_name}', _sub))
for _sub in ['dedup', 'all']:
    _g = IDX_TE if _sub == 'all' else IDX_TE[~META.dup.values[IDX_TE]]
    PROBE_ROWS.append(dict(probe='chance (majority class)', subset=_sub, n_images=len(_g),
                           acc=float(pd.Series(META.label.values[_g]).value_counts(normalize=True).max()),
                           bal_acc=1.0 / N_CLASSES))
T0b = pd.DataFrame(PROBE_ROWS)
T0b.to_csv(os.path.join(TD, 'T0b_metadata_probe.csv'), index=False)
print('\nMetadata classifier (never sees brain pixels):')
print(T0b.round(3).to_string(index=False))

# Figures: source composition + duplicate examples
_fig, _ax = plt.subplots(1, 2, figsize=(9, 2.8))
_src = pd.Series([f'{s} / {"RGB file" if r else "grayscale file"}' for s, r in zip(META.size_group, META.is_rgb)],
                 index=META.index, name='source')
_comp = pd.crosstab(META.cls, _src, normalize='index').reindex(CLASS_NAMES)
_comp.plot.bar(stacked=True, ax=_ax[0], rot=0, colormap='tab20c', width=0.75)
_ax[0].set_ylabel('Fraction of images'); _ax[0].set_xlabel('')
_ax[0].legend(fontsize=5.5, loc='upper left', bbox_to_anchor=(1.0, 1.0))
_d = T0d[['dup_of_train_or_val', 'within_test_extra_copies']]
_d.plot.bar(stacked=True, ax=_ax[1], rot=0, color=['tab:red', 'salmon'], width=0.7)
_ax[1].set_ylabel('Test images removed in dedup'); _ax[1].set_xlabel('')
_ax[1].legend(['copy of a train/val image', 'extra copy within test'], fontsize=6)
_fig.suptitle('BRISC 2025: image source composition and test-set duplicates', fontsize=9)
_fig.tight_layout(rect=(0, 0, 1, 0.92))
save_fig(_fig, os.path.join(FD, 'dataset_audit.pdf'), show=True)

_show = DUPP[(DUPP.pair_type == 'test-trainval') & DUPP.confirmed]
_show = _show.sample(n=min(8, len(_show)), random_state=0) if len(_show) else _show
if len(_show):
    _fig, _ax = plt.subplots(2, len(_show), figsize=(1.6 * len(_show), 3.4))
    _ax = np.array(_ax).reshape(2, -1)
    for _k, _r in enumerate(_show.itertuples()):
        _ax[0, _k].imshow(IMGS[int(_r.a)], cmap='gray'); _ax[0, _k].set_title(f'test: {_r.cls_a}', fontsize=6)
        _ax[1, _k].imshow(IMGS[int(_r.b)], cmap='gray')
        _ax[1, _k].set_title(f'train/val: {_r.cls_b}\nΔpx={_r.pixel_diff:.1f}', fontsize=6)
    for _a in _ax.ravel():
        _a.axis('off')
    _fig.suptitle('Confirmed test–train duplicates (random sample)', fontsize=8)
    _fig.tight_layout(rect=(0, 0, 1, 0.92))
    save_fig(_fig, os.path.join(FD, 'duplicate_examples.pdf'), show=True)

## 3. Synthetic shortcuts (ground truth for validating the audit)

Three conditions, all with **meningioma** as the target class, applied to train, validation and test images:

| Condition | Marker | Prevalence (target / other) | What it tests |
|---|---|---|---|
| `cornerR` | thin "R" text in the top-left corner | 100% / 0% | full shortcut in the background |
| `cornerRp80` | same text | 80% / 20% | **partial** shortcut: the model uses marker and tumour together |
| `arrowHead` | white arrow inside the head, random position and direction, ≥ 10 px from the tumour | 100% / 0% | artefact **inside the anatomy** (like burned-in annotation arrows); invisible to the background arm |

Ground truth is measured directly with the **oracle arm** (occluding exactly the marker vs random translations of
it), because a model trained with a marker does not necessarily use it. Clean ResNet50 models with the same seeds
are audited on the same marked images as negative controls.

In [ ]:
# ===================================================================
# 3) SYNTHETIC SHORTCUTS
# ===================================================================
def draw_corner_text(img, rng):
    """Thin "R" text in the top-left corner (mimics a scanner laterality label)."""
    font, scale, thick = cv2.FONT_HERSHEY_SIMPLEX, 0.8, 1
    (_, th), _ = cv2.getTextSize(SYN_TEXT, font, scale, thick)
    x0, y0 = int(6 + rng.integers(0, 10)), int(6 + th + rng.integers(0, 10))
    canvas = np.zeros_like(img)
    cv2.putText(canvas, SYN_TEXT, (x0, y0), font, scale, 255, thick, cv2.LINE_AA)
    region = cv2.dilate((canvas > 0).astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    return np.maximum(img, canvas), region


def draw_head_arrow(img, fg, tumor, rng):
    """White arrow inside the head; tail ≥ ARROW_LEN+4 px from the head boundary and ≥ 10 px from the tumour."""
    excl = cv2.dilate(tumor.astype(np.uint8), np.ones((21, 21), np.uint8)) > 0
    ok = (fg & ~excl).astype(np.uint8)
    for rad in [ARROW_LEN + 4, ARROW_LEN // 2 + 4]:
        ker = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * rad + 1, 2 * rad + 1))
        ys, xs = np.nonzero(cv2.erode(ok, ker) > 0)
        if len(ys):
            break
    if len(ys) == 0:
        return img, np.zeros(img.shape, bool), False
    i = int(rng.integers(len(ys)))
    ang = float(rng.uniform(0, 2 * np.pi))
    p1 = (int(xs[i]), int(ys[i]))
    p2 = (int(round(xs[i] + ARROW_LEN * np.cos(ang))), int(round(ys[i] + ARROW_LEN * np.sin(ang))))
    canvas = np.zeros_like(img)
    cv2.arrowedLine(canvas, p1, p2, 255, 2, cv2.LINE_AA, 0, 0.45)
    region = cv2.dilate((canvas > 0).astype(np.uint8), np.ones((3, 3), np.uint8)) > 0
    return np.maximum(img, canvas), region, True


def apply_markers(arr, gidx, prob_fn, seed, kind):
    rng = np.random.default_rng(seed)
    out, region, has = arr.copy(), np.zeros(arr.shape, bool), np.zeros(len(arr), bool)
    for g in gidx:
        if rng.random() < prob_fn(g):
            if kind == 'corner_text':
                out[g], region[g] = draw_corner_text(arr[g], rng)
                has[g] = True
            else:
                o, r, ok = draw_head_arrow(arr[g], FG[g], MASKS[g], rng)
                if ok:
                    out[g], region[g], has[g] = o, r, True
    return out, region, has


SYN = {}
if RUN_SYNTHETIC:
    _lab = META.label.values
    for _cond, _cfg in SYN_CONDITIONS.items():
        _imgs, _mark, _has = apply_markers(
            IMGS, np.arange(len(META)),
            lambda g, pt=_cfg['p_target'], po=_cfg['p_other']: pt if _lab[g] == TARGET_IDX else po,
            _cfg['seed'], _cfg['kind'])
        _imgs_all, _, _ = apply_markers(IMGS, IDX_TE, lambda g: 1.0, _cfg['seed'] + 1, _cfg['kind'])
        SYN[_cond] = dict(imgs=_imgs, mark=_mark, has=_has, imgs_all=_imgs_all)
        print(f'Marker prevalence — {_cond}:')
        print(pd.crosstab(META.subset, META.cls, values=_has, aggfunc='mean').reindex(
            index=['train', 'val', 'test'], columns=CLASS_NAMES).round(3), '\n')
    _fig, _ax = plt.subplots(len(SYN_CONDS), 4, figsize=(8, 2.1 * len(SYN_CONDS)))
    _ax = np.atleast_2d(_ax)
    for _r, _cond in enumerate(SYN_CONDS):
        _shown = IDX_TE[(META.label.values[IDX_TE] == TARGET_IDX) & SYN[_cond]['has'][IDX_TE]][:4]
        for _k, _g in enumerate(_shown):
            _ax[_r, _k].imshow(SYN[_cond]['imgs'][_g], cmap='gray')
            _ax[_r, _k].contour(SYN[_cond]['mark'][_g], levels=[0.5], colors='r', linewidths=0.6)
        _ax[_r, 0].set_ylabel(_cond, fontsize=8)
        for _a in _ax[_r]:
            _a.set_xticks([]); _a.set_yticks([])
    _fig.suptitle(f'Synthetic shortcut conditions (target class: {SYN_TARGET})', fontsize=9)
    _fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_fig(_fig, os.path.join(FD, 'synthetic_marker_examples.pdf'), show=True)

## 4. Models and training

Uniform interface for all architectures: feature map `A` (B×C×h×w), `pooled = mean(A)`,
`logits = W·pooled + b`. ImageNet-pretrained backbones, fine-tuned end to end.

In [ ]:
# ===================================================================
# 4) MODEL & TRAINING
# ===================================================================
NORM_MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
NORM_STD = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)


class AuditNet(nn.Module):
    def __init__(self, arch, n_classes, pretrained):
        super().__init__()
        self.arch = arch
        if arch == 'resnet50':
            m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1 if pretrained else None)
            self.body = nn.Sequential(m.conv1, m.bn1, m.relu, m.maxpool, m.layer1, m.layer2, m.layer3, m.layer4)
            dim, p_drop = m.fc.in_features, 0.0
        elif arch == 'densenet121':
            m = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None)
            self.body = nn.Sequential(m.features, nn.ReLU(inplace=False))
            dim, p_drop = m.classifier.in_features, 0.0
        elif arch == 'efficientnet_b0':
            m = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None)
            self.body = m.features
            dim, p_drop = m.classifier[1].in_features, 0.2
        else:
            raise ValueError(f'Unknown architecture: {arch}')
        self.drop = nn.Dropout(p_drop)
        self.fc = nn.Linear(dim, n_classes)
        self.feat_dim = dim
        self.register_buffer('mean', NORM_MEAN.clone())
        self.register_buffer('std', NORM_STD.clone())

    def prep(self, x_gray):                      # (B,1,H,W) in [0,1] → normalised 3-channel input
        return (x_gray.expand(-1, 3, -1, -1) - self.mean) / self.std

    def head(self, pooled):                      # always fp32 → logits = W·pooled + b exactly
        with torch.autocast(device_type=pooled.device.type, enabled=False):
            return self.fc(self.drop(pooled.float()))

    def forward_from_input(self, x3, return_maps=False):
        A = self.body(x3)
        pooled = A.float().mean(dim=(2, 3))
        logits = self.head(pooled)
        return (logits, pooled, A) if return_maps else logits

    def forward(self, x_gray, return_maps=False):
        return self.forward_from_input(self.prep(x_gray), return_maps)


def build_model(arch, pretrained=PRETRAINED):
    try:
        return AuditNet(arch, N_CLASSES, pretrained)
    except Exception as e:
        if pretrained:
            raise RuntimeError(f'Cannot load ImageNet weights for {arch}: {e}\n'
                               '→ Enable Settings → Internet: On in Kaggle and run again.') from e
        raise


def to_batch(arr_u8, idx):
    x = torch.from_numpy(np.ascontiguousarray(arr_u8[idx]))
    return x.to(DEVICE, non_blocking=True).float().div_(255.0).unsqueeze(1)


def gpu_augment(x, gen):
    B = x.shape[0]
    u = torch.rand(B, 3, generator=gen).to(x.device)
    ang = (u[:, 0] * 2 - 1) * math.radians(10.0)
    theta = torch.zeros(B, 2, 3, device=x.device)
    theta[:, 0, 0], theta[:, 0, 1] = torch.cos(ang), -torch.sin(ang)
    theta[:, 1, 0], theta[:, 1, 1] = torch.sin(ang), torch.cos(ang)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode='bilinear', padding_mode='zeros', align_corners=False)
    bright = 1 + (u[:, 1] * 2 - 1) * 0.1
    contr = 1 + (u[:, 2] * 2 - 1) * 0.1
    mean = x.mean(dim=(1, 2, 3), keepdim=True)
    x = ((x - mean) * contr.view(-1, 1, 1, 1) + mean) * bright.view(-1, 1, 1, 1)
    return x.clamp_(0, 1)


@torch.no_grad()
def predict(model, arr, idx, bs=128, want_features=False, want_spatial_mean=False):
    """Logits (+ GAP features, + per-position mean of the spatial feature map A if requested)."""
    model.eval()
    L, P, A_sum = [], [], None
    for s in range(0, len(idx), bs):
        x = to_batch(arr, idx[s:s + bs])
        with amp_ctx():
            logits, pooled, A = model(x, return_maps=True)
        L.append(logits.float().cpu())
        if want_features:
            P.append(pooled.float().cpu())
        if want_spatial_mean:
            a = A.float().sum(0).double()
            A_sum = a if A_sum is None else A_sum + a
    L = torch.cat(L).numpy().astype(np.float64)
    if not want_features:
        return L
    out = (L, torch.cat(P).numpy().astype(np.float64))
    if want_spatial_mean:
        out += ((A_sum / len(idx)).cpu().numpy(),)
    return out


def make_scaler():
    try:
        return torch.amp.GradScaler('cuda', enabled=USE_AMP)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=USE_AMP)


def train_model(arch, seed, arr, idx_tr, idx_va, run_dir):
    ckpt = os.path.join(run_dir, 'model.pt')
    if os.path.exists(ckpt) and not FORCE_RERUN:
        model = build_model(arch, pretrained=False).to(DEVICE)
        model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
        print(f'  checkpoint loaded: {ckpt}')
        return model.eval()
    seed_everything(seed)
    model = build_model(arch).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=STEP_SIZE, gamma=GAMMA)
    scaler = make_scaler()
    gen = torch.Generator().manual_seed(seed)
    y_all = torch.as_tensor(META.label.values, dtype=torch.long)
    y_va = META.label.values[idx_va]
    best, hist = (-1.0, float('inf')), []
    for ep in range(EPOCHS):
        model.train()
        t0 = time.time()
        perm = idx_tr[torch.randperm(len(idx_tr), generator=gen).numpy()]
        tot_loss = tot_ok = tot_n = 0
        for s in range(0, len(perm), BATCH_SIZE):
            bi = perm[s:s + BATCH_SIZE]
            if len(bi) < 2:
                continue
            x = gpu_augment(to_batch(arr, bi), gen)
            y = y_all[bi].to(DEVICE)
            with amp_ctx():
                logits = model(x)
                loss = F.cross_entropy(logits, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            tot_loss += loss.item() * len(bi)
            tot_ok += (logits.argmax(1) == y).sum().item()
            tot_n += len(bi)
        sched.step()
        lv = predict(model, arr, idx_va)
        va_loss = F.cross_entropy(torch.from_numpy(lv), torch.from_numpy(y_va)).item()
        va_acc = float((lv.argmax(1) == y_va).mean())
        improved = va_acc > best[0] or (va_acc == best[0] and va_loss < best[1])
        if improved:
            best = (va_acc, va_loss)
            torch.save(model.state_dict(), ckpt)
        hist.append(dict(epoch=ep + 1, train_loss=tot_loss / max(tot_n, 1), train_acc=tot_ok / max(tot_n, 1),
                         val_loss=va_loss, val_acc=va_acc, saved=improved, sec=time.time() - t0))
        print(f'  epoch {ep + 1:2d}/{EPOCHS} | train loss {hist[-1]["train_loss"]:.4f} acc {hist[-1]["train_acc"]:.4f}'
              f' | val loss {va_loss:.4f} acc {va_acc:.4f}{" *" if improved else ""} | {hist[-1]["sec"]:.0f}s')
    pd.DataFrame(hist).to_csv(os.path.join(run_dir, 'train_log.csv'), index=False)
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE, weights_only=True))
    return model.eval()


if PRETRAINED:
    for _a in dict.fromkeys(ARCHS + [SYN_ARCH]):
        build_model(_a)
    print('ImageNet weights available for:', ', '.join(dict.fromkeys(ARCHS + [SYN_ARCH])))


def classification_report(L, y):
    pred = L.argmax(1)
    cm = np.zeros((N_CLASSES, N_CLASSES), int)
    np.add.at(cm, (y, pred), 1)
    rec = np.diag(cm) / np.maximum(cm.sum(1), 1)
    prec = np.diag(cm) / np.maximum(cm.sum(0), 1)
    f1 = 2 * prec * rec / np.maximum(prec + rec, 1e-12)
    return dict(n=int(len(y)), acc=float((pred == y).mean()), bal_acc=float(rec.mean()), macro_f1=float(f1.mean()),
                recall={CLASS_NAMES[c]: float(rec[c]) for c in range(N_CLASSES)}, confusion=cm.tolist())

## 4b. Fingerprint probes: class information outside the lesion

Linear probes (StandardScaler → PCA 256 → logistic regression) on features of a **frozen ImageNet ResNet50**
(not fine-tuned), trained on the training split and evaluated on the deduplicated test set. Because the features
are frozen, probe accuracy is a **lower bound** on the available information.

| Probe | Input | What it measures |
|---|---|---|
| full image | whole image | reference: total information readable by a linear probe |
| tumour-free patches | 4 patches of 56 px per image, ≥ 95% inside the head, ≥ 24 px from the tumour | class information outside the lesion: anatomical context + acquisition characteristics |
| tumour-free noise residual | high-pass residual (σ = 1.5) of the same patches, normalised | mainly noise / compression texture; most anatomy removed |
| background only | image with the head (dilated by 5 px) set to black | information in non-anatomical areas |

For no_tumour images, the exclusion zone is taken from the tumour mask of another image of the same split, so that
patch locations are comparable across classes. Patches of gliomas and meningiomas may still contain secondary signs
(oedema, mass effect); the cleanest interpretation is for pituitary (small sellar lesion) and no_tumour.

In [ ]:
# ===================================================================
# 4b) FINGERPRINT PROBES
# ===================================================================
def integral(a):
    return np.pad(a.astype(np.int32).cumsum(0).cumsum(1), ((1, 0), (1, 0)))


def window_sums(ii, P):
    return ii[P:, P:] - ii[:-P, P:] - ii[P:, :-P] + ii[:-P, :-P]


FP_KER = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * FP_EXCL_DIL + 1, 2 * FP_EXCL_DIL + 1))
FP_POOLS = {s: np.flatnonzero((META.subset.values == s) & META.mask_ok.values) for s in ['train', 'val', 'test']}


def fp_exclusion(g, rng):
    if META.label.values[g] != 0:
        if not META.mask_ok.values[g]:
            return None
        m = MASKS[g]
    else:
        pool = FP_POOLS[META.subset.values[g]]
        if len(pool) == 0:
            return np.zeros(FG[g].shape, bool)
        m = MASKS[pool[int(rng.integers(len(pool)))]]
    return cv2.dilate(m.astype(np.uint8), FP_KER) > 0


def fp_positions(g, excl, rng):
    P = FP_PATCH
    fg_s = window_sums(integral(FG[g]), P)
    ex_s = window_sums(integral(excl), P)
    ys, xs = np.nonzero((fg_s >= FP_MIN_FG * P * P) & (ex_s == 0))
    if len(ys) == 0:
        return []
    pick = rng.choice(len(ys), size=min(FP_PER_IMAGE, len(ys)), replace=False)
    return [(int(ys[i]), int(xs[i])) for i in pick]


def fp_residual(p):
    r = p - cv2.GaussianBlur(p, (0, 0), FP_RESID_SIGMA)
    z = (r - r.mean()) / (r.std() + 1e-6)
    return ((np.clip(z, -3, 3) + 3) / 6).astype(np.float32)


@torch.no_grad()
def fp_features(body, arr):
    """arr: (n,H,W) float32 in [0,1] → 2048-d GAP features from a frozen ImageNet ResNet50."""
    out = []
    mean, std = NORM_MEAN.to(DEVICE), NORM_STD.to(DEVICE)
    for s in range(0, len(arr), 256):
        x = torch.from_numpy(np.ascontiguousarray(arr[s:s + 256], dtype=np.float32)).to(DEVICE)[:, None]
        x = (x.expand(-1, 3, -1, -1) - mean) / std
        with amp_ctx():
            A = body(x)
        out.append(A.float().mean((2, 3)).cpu().numpy())
    return np.concatenate(out) if out else np.zeros((0, 2048), np.float32)


def fp_patch_features(body, gidx, seed):
    rng = np.random.default_rng(seed)
    F_raw, F_res, owner = [], [], []
    for s in range(0, len(gidx), 500):
        pr, pz, own = [], [], []
        for g in gidx[s:s + 500]:
            ex = fp_exclusion(g, rng)
            if ex is None:
                continue
            for (y, x) in fp_positions(g, ex, rng):
                p = IMGS[g, y:y + FP_PATCH, x:x + FP_PATCH].astype(np.float32) / 255.0
                pr.append(cv2.resize(p, (FP_INPUT, FP_INPUT), interpolation=cv2.INTER_LINEAR))
                pz.append(cv2.resize(fp_residual(p), (FP_INPUT, FP_INPUT), interpolation=cv2.INTER_LINEAR))
                own.append(g)
        if own:
            F_raw.append(fp_features(body, np.stack(pr)))
            F_res.append(fp_features(body, np.stack(pz)))
            owner += own

    def cat(L):
        return np.concatenate(L) if L else np.zeros((0, 2048), np.float32)
    return cat(F_raw), cat(F_res), np.asarray(owner, dtype=int)


def fp_image_features(body, gidx, background_only=False):
    out = []
    for s in range(0, len(gidx), 256):
        g = gidx[s:s + 256]
        a = IMGS[g].astype(np.float32) / 255.0
        if background_only:
            keep = np.stack([~(cv2.dilate(FG[k].astype(np.uint8), DIL11) > 0) for k in g])
            a = a * keep
        out.append(fp_features(body, a))
    return np.concatenate(out)


def fp_fit_predict(Xtr, ytr, Xte, gte):
    """Fit the probe; predict per image (mean log-probability over that image's patches)."""
    n_comp = int(min(FP_PCA, Xtr.shape[0] - 1, Xtr.shape[1]))
    pipe = make_pipeline(StandardScaler(), PCA(n_components=n_comp, random_state=0),
                         LogisticRegression(C=0.1, max_iter=3000))
    pipe.fit(Xtr, ytr)
    LP = pipe.predict_log_proba(Xte)
    uniq, inv = np.unique(np.asarray(gte, dtype=int), return_inverse=True)
    S = np.zeros((len(uniq), LP.shape[1]))
    np.add.at(S, inv.ravel(), LP)
    return uniq, np.asarray(pipe.classes_, dtype=int)[S.argmax(1)]


f_fp = os.path.join(TD, 'T8_fingerprint_probes.csv')
if RUN_FINGERPRINT and PRETRAINED and (FORCE_RERUN or not os.path.exists(f_fp)):
    t0 = time.time()
    _m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)
    FP_BODY = nn.Sequential(_m.conv1, _m.bn1, _m.relu, _m.maxpool, _m.layer1, _m.layer2, _m.layer3, _m.layer4).to(DEVICE).eval()
    _all = np.arange(len(META))
    _Ffull = fp_image_features(FP_BODY, _all)
    _Fbg = fp_image_features(FP_BODY, _all, background_only=True)
    _Ptr_raw, _Ptr_res, _Otr = fp_patch_features(FP_BODY, IDX_TR, seed=0)
    _Pte_raw, _Pte_res, _Ote = fp_patch_features(FP_BODY, IDX_TE, seed=1)
    _ytr_img = META.label.values[IDX_TR]
    _rows = [r for r in PROBE_ROWS]
    for _name, (_Xtr, _ytr, _Xte, _gte) in {
            'full image (frozen ResNet50)': (_Ffull[IDX_TR], _ytr_img, _Ffull[IDX_TE], IDX_TE),
            'tumour-free patches': (_Ptr_raw, META.label.values[_Otr], _Pte_raw, _Ote),
            'tumour-free noise residual': (_Ptr_res, META.label.values[_Otr], _Pte_res, _Ote),
            'background only': (_Fbg[IDX_TR], _ytr_img, _Fbg[IDX_TE], IDX_TE)}.items():
        if len(_Xtr) < 20 or len(_Xte) == 0:
            print(f'  {_name}: not enough data, skipped')
            continue
        _g, _p = fp_fit_predict(_Xtr, _ytr, _Xte, _gte)
        for _sub in ['dedup', 'all']:
            _rows.append(score_predictions(_p, _g, _name, _sub))
    T8 = pd.DataFrame(_rows)
    T8.to_csv(f_fp, index=False)
    print(f'Fingerprint probes finished in {(time.time() - t0) / 60:.1f} min '
          f'(train patches: {len(_Otr)}, test patches: {len(_Ote)} from {len(np.unique(_Ote))} images)')
    del FP_BODY, _m, _Ffull, _Fbg, _Ptr_raw, _Ptr_res, _Pte_raw, _Pte_res
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
elif os.path.exists(f_fp):
    T8 = pd.read_csv(f_fp)
else:
    T8 = pd.DataFrame(PROBE_ROWS)
if len(T8):
    _t = T8[T8.subset == 'dedup']
    print('Class-information probes (deduplicated test set):')
    print(_t.round(3).to_string(index=False))
    _plot = _t[~_t.probe.str.startswith('chance')]
    if len(_plot):
        _fig, _ax = plt.subplots(figsize=(9, 3.0))
        _cols = [f'recall_{c}' for c in CLASS_NAMES] + ['bal_acc']
        _vals = _plot.set_index('probe')[_cols]
        _vals.columns = [c.replace('recall_', '') for c in CLASS_NAMES] + ['balanced acc.']
        _vals.plot.bar(ax=_ax, rot=20, width=0.8, colormap='tab10')
        _ax.axhline(1.0 / N_CLASSES, color='k', ls='--', lw=0.8, label=f'chance ({N_CLASSES} classes)')
        _ax.set_ylabel('Test recall / balanced accuracy (dedup)'); _ax.set_xlabel('')
        _ax.set_ylim(0, 1.05)
        _ax.legend(fontsize=6, ncol=3)
        plt.setp(_ax.get_xticklabels(), fontsize=6, ha='right')
        _ax.set_title('How much class information is available outside the lesion?', fontsize=9)
        save_fig(_fig, os.path.join(FD, 'fingerprint_probes.pdf'), show=True)

## 5. Spectral analysis (fitted on training features)

Eigen-decomposition of the GAP features with three estimates of the number of informative components
(Marchenko–Pastur edge with Tracy–Widom correction, Gavish–Donoho, exact parallel analysis) and fidelity controls.
The selected components define the `spectral` and `spectral_subspace` maps.

In [ ]:
# ===================================================================
# 5) SPECTRAL ANALYSIS
# ===================================================================
def bh(p):
    p = np.asarray(p, float)
    out = np.full(p.shape, np.nan)
    ok = np.isfinite(p)
    if ok.sum() == 0:
        return out
    pv = p[ok]
    order = np.argsort(pv)
    ranked = pv[order] * len(pv) / np.arange(1, len(pv) + 1)
    q = np.clip(np.minimum.accumulate(ranked[::-1])[::-1], 0, 1)
    tmp = np.empty(len(pv))
    tmp[order] = q
    out[ok] = tmp
    return out


def mp_std_median(gamma, n=400001):
    """Median of the standard MP distribution (σ² = 1) with ratio gamma ≤ 1."""
    lo, hi = (1 - math.sqrt(gamma)) ** 2, (1 + math.sqrt(gamma)) ** 2
    x = np.linspace(lo, hi, n)[1:-1]
    dens = np.sqrt((hi - x) * (x - lo)) / (2 * np.pi * gamma * x)
    cdf = np.cumsum(dens)
    return float(np.interp(0.5, cdf / cdf[-1], x))


def mp_density(lam, sigma2, q):
    """Density of the non-zero sample-covariance eigenvalues under pure noise (ratio q = D/N)."""
    lam = np.asarray(lam, float)
    if q <= 1:
        lo, hi = sigma2 * (1 - math.sqrt(q)) ** 2, sigma2 * (1 + math.sqrt(q)) ** 2
        return np.sqrt(np.clip((hi - lam) * (lam - lo), 0, None)) / (2 * np.pi * sigma2 * q * np.maximum(lam, 1e-300))
    s, g = q * sigma2, 1 / q
    x = lam / s
    lo, hi = (1 - math.sqrt(g)) ** 2, (1 + math.sqrt(g)) ** 2
    return np.sqrt(np.clip((hi - x) * (x - lo), 0, None)) / (2 * np.pi * g * np.maximum(x, 1e-300)) / s


def spectral_fit(Ftr, seed):
    N, D = Ftr.shape
    mu = Ftr.mean(0)
    Xc = Ftr - mu
    eig, V = np.linalg.eigh(Xc.T @ Xc / (N - 1))
    order = np.argsort(eig)[::-1]
    eig, V = np.clip(eig[order], 0, None), V[:, order]
    m = min(D, N - 1)
    eig = eig[:m]
    n_eff = N - 1
    q = D / n_eff
    mu_np = (math.sqrt(n_eff - 1) + math.sqrt(D)) ** 2
    sd_np = (math.sqrt(n_eff - 1) + math.sqrt(D)) * (1 / math.sqrt(n_eff - 1) + 1 / math.sqrt(D)) ** (1 / 3)

    def tw_edge(s2):
        return s2 * (mu_np + 0.9793 * sd_np) / n_eff

    med_std = mp_std_median(q) if q <= 1 else q * mp_std_median(1 / q)
    sigma2_med = float(np.median(eig) / med_std)
    bulk = eig[eig <= tw_edge(sigma2_med)]
    sigma2 = float(bulk.mean() / (1.0 if q <= 1 else q)) if len(bulk) else sigma2_med
    edge = tw_edge(sigma2)
    edge_asym = sigma2 * (1 + math.sqrt(q)) ** 2
    K_mp = int((eig > edge).sum())
    sv = np.sqrt(eig * n_eff)
    beta = min(N, D) / max(N, D)
    tau_sv = (0.56 * beta ** 3 - 0.95 * beta ** 2 + 1.82 * beta + 1.43) * float(np.median(sv))
    K_gd = int((sv > tau_sv).sum())
    n_comp = min(PA_COMPONENTS, m)
    rng = np.random.default_rng(seed)
    X32 = Xc.astype(np.float32)
    null = np.zeros((PA_PERMUTATIONS, n_comp))
    for p in range(PA_PERMUTATIONS):
        Xp = np.take_along_axis(X32, rng.random(X32.shape).argsort(axis=0), axis=0)
        G = (Xp.T @ Xp) if D <= N else (Xp @ Xp.T)
        null[p] = np.linalg.eigvalsh(G.astype(np.float64))[::-1][:n_comp] / n_eff
    k_ord = min(PA_PERMUTATIONS, math.ceil(0.95 * (PA_PERMUTATIONS + 1))) - 1
    pa_thr = np.sort(null, axis=0)[k_ord]
    above = eig[:n_comp] > pa_thr
    K_pa = int(n_comp if above.all() else np.argmin(above))
    k_keep = min(max(K_mp, K_gd, K_pa, 10), m)
    Vk = V[:, :k_keep].copy()
    Vk *= np.sign(Vk[np.abs(Vk).argmax(0), np.arange(k_keep)])
    return dict(N=N, D=D, mu=mu, V=Vk, eig=eig, sigma2=sigma2, sigma2_median=sigma2_med, edge=edge,
                edge_asym=edge_asym, q=q,
                K_mp=K_mp, K_mp_asym=int((eig > edge_asym).sum()),
                K_mp_median=int((eig > tw_edge(sigma2_med)).sum()), K_gd=K_gd, K_pa=K_pa,
                tau_gd_eig=tau_sv ** 2 / n_eff, pa_thr=pa_thr)


def fidelity_analysis(Ftr, ytr, Fte, yte, W, b, spec, K, seed):
    mu, VK = spec['mu'], spec['V'][:, :K]
    Xtr, Xte = Ftr - mu, Fte - mu

    def acc_of(F_):
        return float(((F_ @ W.T + b).argmax(1) == yte).mean())

    full = Fte @ W.T + b
    Pte = (Xte @ VK) @ VK.T
    top = (mu + Pte) @ W.T + b
    Qw, _ = np.linalg.qr(W.T)
    rng = np.random.default_rng(seed)
    acc_rand = []
    for _ in range(N_RANDOM_SUBSPACE):
        Q, _ = np.linalg.qr(rng.standard_normal((W.shape[1], K)))
        acc_rand.append(acc_of(mu + (Xte @ Q) @ Q.T))
    cf, ct = full.argmax(1) == yte, top.argmax(1) == yte
    b01, b10 = int((cf & ~ct).sum()), int((~cf & ct).sum())
    p_mcn = float(stats.binomtest(min(b01, b10), b01 + b10, 0.5).pvalue) if b01 + b10 > 0 else 1.0
    Wc = W - W.mean(0, keepdims=True)

    def probe(A, B_):
        sc = StandardScaler().fit(A)
        clf = LogisticRegression(C=0.1, max_iter=2000).fit(sc.transform(A), ytr)
        return float((clf.predict(sc.transform(B_)) == yte).mean())

    Ptr = (Xtr @ VK) @ VK.T
    return dict(
        K=K, var_frac_topK=float(spec['eig'][:K].sum() / spec['eig'].sum()),
        weight_energy_topK=float(np.linalg.norm(Wc @ VK) ** 2 / np.linalg.norm(Wc) ** 2),
        logit_R2_topK=float(1 - ((full - top) ** 2).sum() / ((full - full.mean(0)) ** 2).sum()),
        acc_full=float(cf.mean()), acc_topK=float(ct.mean()),
        agree_topK=float((full.argmax(1) == top.argmax(1)).mean()), mcnemar_p=p_mcn,
        acc_complement=acc_of(Fte - Pte), acc_spanW=acc_of(mu + (Xte @ Qw) @ Qw.T),
        acc_randK_mean=float(np.mean(acc_rand)), acc_randK_sd=float(np.std(acc_rand)),
        probe_full=probe(Xtr, Xte), probe_topK=probe(Xtr @ VK, Xte @ VK),
        probe_complement=probe(Xtr - Ptr, Xte - Pte),
        chance=float(np.bincount(yte, minlength=N_CLASSES).max() / len(yte)))


def component_analysis(spec, K, W, Ftr, ytr, Fte, yte):
    """Select one component per class on TRAINING data: largest contribution to that class's differential logit."""
    mu, VK = spec['mu'], spec['V'][:, :K]
    Wd = np.stack([W[c] - np.delete(W, c, axis=0).mean(0) for c in range(N_CLASSES)])
    Utr, Ute = (Ftr - mu) @ VK, (Fte - mu) @ VK
    WdV = Wd @ VK
    S = np.stack([(Utr[ytr == c] * WdV[c]).mean(0) for c in range(N_CLASSES)])
    sel = S.argmax(1)
    H, P = [], []
    for k in range(K):
        groups = [Ute[yte == c, k] for c in range(N_CLASSES) if (yte == c).sum() > 1]
        h, p = stats.kruskal(*groups) if len(groups) > 1 else (np.nan, np.nan)
        H.append(h); P.append(p)
    tab = pd.DataFrame({'component': np.arange(K), 'eigenvalue': spec['eig'][:K],
                        'var_frac': spec['eig'][:K] / spec['eig'].sum(), 'above_MP': spec['eig'][:K] > spec['edge'],
                        'KW_H_test': H, 'epsilon2_test': np.asarray(H) / (len(yte) - 1), 'q_KW_test': bh(P)})
    for c in range(N_CLASSES):
        tab[f'WdV_{CLASS_NAMES[c]}'] = WdV[c]
        tab[f'contrib_train_{CLASS_NAMES[c]}'] = S[c]
    tab['selected_for'] = [','.join(CLASS_NAMES[c] for c in range(N_CLASSES) if sel[c] == k) for k in range(K)]
    return tab, sel, S, Wd, Ute, WdV


def plot_spectrum(spec, title, path, show=False):
    eig, n = spec['eig'], min(len(spec['eig']), 120)
    fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
    ax[0].semilogy(np.arange(1, n + 1), eig[:n], 'o-', ms=2, lw=0.7, color='k', label='Eigenvalue (train)')
    ax[0].axhline(spec['edge'], color='tab:red', ls='--', lw=1, label=f'MP edge + TW 95% (K={spec["K_mp"]})')
    ax[0].axhline(spec['tau_gd_eig'], color='tab:blue', ls=':', lw=1, label=f'Gavish–Donoho (K={spec["K_gd"]})')
    pa = spec['pa_thr'][:n]
    ax[0].semilogy(np.arange(1, len(pa) + 1), pa, color='tab:green', lw=1, label=f'Parallel analysis 95% (K={spec["K_pa"]})')
    ax[0].set_xlabel('Component'); ax[0].set_ylabel('Eigenvalue'); ax[0].legend(fontsize=6)
    bulk = eig[(eig > 0) & (eig <= 3 * spec['edge'])]
    if len(bulk) > 5:
        ax[1].hist(bulk, bins=80, density=True, color='0.75', label='Empirical bulk')
        lam = np.linspace(bulk.min(), bulk.max(), 400)
        ax[1].plot(lam, mp_density(lam, spec['sigma2'], spec['q']), color='tab:red', lw=1.2, label='MP fit')
        ax[1].axvline(spec['edge'], color='tab:red', ls='--', lw=0.8)
        ax[1].set_xlabel('Eigenvalue'); ax[1].set_ylabel('Density'); ax[1].legend(fontsize=6)
    fig.suptitle(title, fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_fig(fig, path, show)


def plot_components(Ute, yte, sel, WdV, title, path, show=False):
    fig, ax = plt.subplots(1, N_CLASSES, figsize=(2.6 * N_CLASSES, 2.6), sharey=False)
    for c in range(N_CLASSES):
        k = sel[c]
        score = Ute[:, k] * np.sign(WdV[c, k])
        ax[c].boxplot([score[yte == j] for j in range(N_CLASSES)], showfliers=False)
        ax[c].set_xticks(range(1, N_CLASSES + 1))
        ax[c].set_xticklabels([n[:4] for n in CLASS_NAMES], fontsize=7)
        ax[c].set_title(f'{CLASS_NAMES[c]}: component {k}', fontsize=8)
    ax[0].set_ylabel('Oriented component score (test)')
    fig.suptitle(title, fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_fig(fig, path, show)

## 6. Explanation maps

| Map | Definition | Native resolution |
|---|---|---|
| `spectral` | (W̃_c·v_k)·v_kᵀ(A_ij − μ_ij), k = component selected for class c | 7×7 |
| `spectral_subspace` | W̃_cᵀ P_K (A_ij − μ_ij) | 7×7 |
| `cam` | W_c·A_ij (= last-layer Grad-CAM) | 7×7 |
| `eigencam` | per-image first principal component, sign-aligned | 7×7 |
| `ig` | Integrated Gradients of the differential logit (logit_c − mean of the other logits), black baseline, 20 steps, Gaussian smoothing σ = 4 px | 224×224 |

W̃_c = W_c − mean of the other rows of W; μ_ij = mean training feature map at position (i, j). All maps are signed
(no ReLU) before ranking; positive values are evidence for class c. IG uses manual loss scaling so fp16 gradients do
not underflow, and is recomputed in fp32 if any value is non-finite.

In [ ]:
# ===================================================================
# 6) EXPLANATION MAPS
# ===================================================================
def make_map_ctx(spec, K, sel, Wd, W, mu_sp):
    """mu_sp: per-position mean training feature map (C,h,w); its spatial mean equals μ (mean GAP feature)."""
    V = spec['V']
    g_spec = np.stack([(Wd[c] @ V[:, sel[c]]) * V[:, sel[c]] for c in range(N_CLASSES)])
    VK = V[:, :K]
    g_sub = (Wd @ VK) @ VK.T

    def t(a):
        return torch.tensor(np.asarray(a), dtype=torch.float32, device=DEVICE)

    return {'g_spectral': t(g_spec), 'c_spectral': t(np.einsum('kc,chw->khw', g_spec, mu_sp)),
            'g_spectral_subspace': t(g_sub), 'c_spectral_subspace': t(np.einsum('kc,chw->khw', g_sub, mu_sp)),
            'g_cam': t(W), 'c_cam': t(np.zeros((N_CLASSES,) + mu_sp.shape[1:]))}


def class_map(kind, A, y, ctx):
    """A: (B,C,h,w) float32, y: (B,) → signed map (B,H,W) at image resolution."""
    if kind == 'eigencam':
        B, C, h, w = A.shape
        X = A.reshape(B, C, h * w).transpose(1, 2)
        Xc = X - X.mean(1, keepdim=True)
        _, _, Vh = torch.linalg.svd(Xc, full_matrices=False)
        proj = (Xc @ Vh[:, 0, :].unsqueeze(-1)).squeeze(-1)
        ref = X.mean(2)
        sgn = torch.sign(((proj - proj.mean(1, keepdim=True)) * (ref - ref.mean(1, keepdim=True))).sum(1))
        sgn[sgn == 0] = 1
        M = (proj * sgn[:, None]).reshape(B, h, w)
    else:
        M = torch.einsum('bc,bchw->bhw', ctx['g_' + kind][y], A) - ctx['c_' + kind][y]
    return F.interpolate(M[:, None], size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False)[:, 0]


def _ig_attribution(model, x, y, steps, use_amp):
    w = F.one_hot(y, N_CLASSES).float() * (1 + 1 / (N_CLASSES - 1)) - 1 / (N_CLASSES - 1)   # logit_y − mean of the other logits
    scale = 256.0 if use_amp else 1.0
    total = torch.zeros_like(x)
    for k in range(steps):
        xi = (x * ((k + 0.5) / steps)).detach().requires_grad_(True)
        ctx = torch.autocast(device_type='cuda', dtype=torch.float16) if use_amp else contextlib.nullcontext()
        with ctx:
            logits = model(xi)
        target = (logits.float() * w).sum() * scale
        g, = torch.autograd.grad(target, xi)
        total += g.float() / scale
    return x * total / steps


def ig_map(model, x, y, steps=None):
    """Integrated Gradients (black baseline) → signed map (B,H,W), Gaussian-smoothed."""
    model.eval()
    steps = steps or IG_STEPS
    with torch.enable_grad():
        att = _ig_attribution(model, x, y, steps, USE_AMP)
        if USE_AMP and not torch.isfinite(att).all():
            att = _ig_attribution(model, x, y, steps, False)
    att = torch.nan_to_num(att.detach(), nan=0.0, posinf=0.0, neginf=0.0)
    return gblur(att, IG_SMOOTH_SIGMA)[:, 0]

## 7. Area-matched three-arm audit with shape-matched null distributions

- **Arm 1 (map region)**: the top-k pixels of the explanation map, with k = budget × tumour area.
- **Arm 3 (tumour)**: the tumour mask (budget 1×) or the tumour plus its nearest pixels up to the same area (2×).
- **Null distributions**: each region is compared with N_RAND = 30 random translations of the same shape that overlap
  the tumour ≤ 10%, overlap the original region ≤ 10% and keep the head-foreground fraction within ±0.15.
- **Map-free arms**: the background arm (everything outside the dilated head set to black) and, for synthetic data,
  the oracle arm (exact marker region). On synthetic data the background arm is also run on the same image without
  the marker, to isolate background reliance caused by the marker.

In [ ]:
# ===================================================================
# 7) ABLATION
# ===================================================================
def gauss1d(sigma, device):
    r = max(1, int(math.ceil(3 * sigma)))
    x = torch.arange(-r, r + 1, device=device, dtype=torch.float32)
    k = torch.exp(-x ** 2 / (2 * sigma ** 2))
    return k / k.sum(), r


def gblur(x, sigma):
    k, r = gauss1d(sigma, x.device)
    x = F.conv2d(F.pad(x, (r, r, 0, 0)), k.view(1, 1, 1, -1))
    return F.conv2d(F.pad(x, (0, 0, r, r)), k.view(1, 1, -1, 1))


def nconv_fill(x, m, sigma=NCONV_SIGMA, down=4):
    keep = 1.0 - m
    num = gblur(F.avg_pool2d(x * keep, down), sigma / down)
    den = gblur(F.avg_pool2d(keep, down), sigma / down)
    fill = F.interpolate(num / den.clamp_min(1e-6), size=x.shape[-2:], mode='bilinear', align_corners=False)
    den_up = F.interpolate(den, size=x.shape[-2:], mode='bilinear', align_corners=False)
    gmean = (x * keep).sum((2, 3), keepdim=True) / keep.sum((2, 3), keepdim=True).clamp_min(1.0)
    return torch.where(den_up > 1e-3, fill, gmean.expand_as(fill))


def fill_image(x, m, mode):
    if mode == 'nconv':
        return nconv_fill(x, m)
    if mode == 'blur':
        return gblur(x, 8.0)
    if mode == 'black':
        return torch.zeros_like(x)
    raise ValueError(f'Unknown FILL_MODE: {mode}')


@torch.no_grad()
def forward_ablated(model, x, masks, owner, fill_mode):
    """x: (B,1,H,W); masks: (M,H,W) bool; owner: (M,) image index → logits (M,C)."""
    out = []
    for s in range(0, masks.shape[0], FORWARD_CHUNK):
        o = owner[s:s + FORWARD_CHUNK]
        m = masks[s:s + FORWARD_CHUNK][:, None].float()
        xi = x[o]
        if fill_mode == 'imagenet_mean':
            x3 = model.prep(xi) * (1 - m)
            with amp_ctx():
                lg = model.forward_from_input(x3)
        else:
            xa = xi * (1 - m) + fill_image(xi, m, fill_mode) * m
            with amp_ctx():
                lg = model(xa)
        out.append(lg.float())
    return torch.cat(out) if out else torch.zeros(0, N_CLASSES, device=x.device)


def margins(logits, y):
    lt = logits.gather(1, y[:, None])[:, 0]
    other = logits.masked_fill(F.one_hot(y, logits.shape[1]).bool(), float('-inf')).max(1).values
    return lt - other, lt


def topk_mask(M, areas, gen):
    B = M.shape[0]
    flat = M.reshape(B, -1)
    noise = torch.randn(flat.shape, generator=gen).to(flat.device) * (flat.std(1, keepdim=True) * 1e-6 + 1e-12)
    order = torch.argsort(flat + noise, dim=1, descending=True)
    ranks = torch.empty_like(order)
    ranks.scatter_(1, order, torch.arange(flat.shape[1], device=flat.device).expand(B, -1).contiguous())
    return (ranks < areas[:, None]).reshape(M.shape)


def dilate(m, r):
    return F.max_pool2d(m[:, None].float(), kernel_size=2 * r + 1, stride=1, padding=r)[:, 0] > 0


def sample_shifts(R, T, FGm, n, gen):
    """Valid random translations for each region R (B,H,W). Returns a list of (dy, dx) per image."""
    B, H, W = R.shape
    P = (2 * H, 2 * W)
    Rf = R.float()
    a = Rf.sum((1, 2))
    FR = torch.fft.rfft2(Rf, s=P)

    def xcorr(Z):
        return torch.round(torch.fft.irfft2(torch.conj(FR) * torch.fft.rfft2(Z.float(), s=P), s=P))

    ov_t, ov_fg, ov_self = xcorr(T), xcorr(FGm), xcorr(R)
    fg0 = (R & FGm).float().sum((1, 2)) / a.clamp_min(1)
    dys = torch.arange(P[0], device=R.device)
    dys = torch.where(dys < H, dys, dys - P[0])
    dxs = torch.arange(P[1], device=R.device)
    dxs = torch.where(dxs < W, dxs, dxs - P[1])
    rows, cols = R.any(2), R.any(1)
    r0 = rows.float().argmax(1)
    r1 = H - 1 - rows.flip(1).float().argmax(1)
    c0 = cols.float().argmax(1)
    c1 = W - 1 - cols.flip(1).float().argmax(1)
    inb_y = (dys[None] >= -r0[:, None]) & (dys[None] <= (H - 1 - r1)[:, None])
    inb_x = (dxs[None] >= -c0[:, None]) & (dxs[None] <= (W - 1 - c1)[:, None])
    aa = a[:, None, None]
    valid = inb_y[:, :, None] & inb_x[:, None, :]
    valid &= ov_t <= MAX_TUMOR_OVERLAP * aa + 0.5
    valid &= ov_self <= MAX_SELF_OVERLAP * aa + 0.5
    valid &= (ov_fg / aa.clamp_min(1) - fg0[:, None, None]).abs() <= FG_TOL
    valid &= (a > 0)[:, None, None]
    flat = valid.reshape(B, -1)
    out = []
    for i in range(B):
        idx = torch.nonzero(flat[i]).squeeze(1)
        k = min(n, idx.numel())
        if k == 0:
            out.append((np.zeros(0, int), np.zeros(0, int)))
            continue
        pick = idx[torch.randperm(idx.numel(), generator=gen)[:k].to(idx.device)]
        out.append(((dys[pick // P[1]]).cpu().numpy(), (dxs[pick % P[1]]).cpu().numpy()))
    return out


def build_shifted(R, shifts):
    masks, owner = [], []
    for i, (dy, dx) in enumerate(shifts):
        for a_, b_ in zip(dy, dx):
            masks.append(torch.roll(R[i], shifts=(int(a_), int(b_)), dims=(0, 1)))
            owner.append(i)
    if not masks:
        return torch.zeros((0,) + tuple(R.shape[1:]), dtype=torch.bool, device=R.device), \
            torch.zeros(0, dtype=torch.long, device=R.device)
    return torch.stack(masks), torch.tensor(owner, dtype=torch.long, device=R.device)


def eval_region_and_null(model, x, y, M0, L0, pred0, R, valid_rows, Tm, FGm, fill_mode, gen):
    """Ablate region R and N_RAND translations of it. Per-image results + trial matrix (B, N_RAND)."""
    B = R.shape[0]
    keys = ['dM', 'dL', 'flip', 'dM_rand_mean', 'dM_rand_std', 'dL_rand_mean', 'flip_rand', 'n_rand', 'p_emp']
    out = {k: np.full(B, np.nan) for k in keys}
    out['n_rand'][:] = 0
    tr = np.full((B, N_RAND), np.nan, np.float32)
    rows = torch.nonzero(valid_rows).squeeze(1)
    if rows.numel() == 0:
        return out, tr, [(np.zeros(0, int), np.zeros(0, int))] * B
    r_np = rows.cpu().numpy()
    lg = forward_ablated(model, x, R[rows], rows, fill_mode)
    m_, l_ = margins(lg, y[rows])
    out['dM'][r_np] = (M0[rows] - m_).cpu().numpy()
    out['dL'][r_np] = (L0[rows] - l_).cpu().numpy()
    out['flip'][r_np] = (lg.argmax(1) != pred0[rows]).float().cpu().numpy()
    shifts = sample_shifts(R[rows], Tm[rows], FGm[rows], N_RAND, gen)
    Rr, own_local = build_shifted(R[rows], shifts)
    if own_local.numel():
        own = rows[own_local]
        lg2 = forward_ablated(model, x, Rr, own, fill_mode)
        m2, l2 = margins(lg2, y[own])
        dM2, dL2 = (M0[own] - m2).cpu().numpy(), (L0[own] - l2).cpu().numpy()
        fl2 = (lg2.argmax(1) != pred0[own]).float().cpu().numpy()
        prim = dM2 if PRIMARY == 'dM' else dL2
        own_np = own.cpu().numpy()
        for j in np.unique(own_np):
            sel = own_np == j
            v = prim[sel]
            tr[j, :len(v)] = v
            out['dM_rand_mean'][j] = dM2[sel].mean()
            out['dM_rand_std'][j] = dM2[sel].std(ddof=1) if sel.sum() > 1 else np.nan
            out['dL_rand_mean'][j] = dL2[sel].mean()
            out['flip_rand'][j] = fl2[sel].mean()
            out['n_rand'][j] = sel.sum()
            obs = out['dM'][j] if PRIMARY == 'dM' else out['dL'][j]
            out['p_emp'][j] = (1 + (v >= obs).sum()) / (len(v) + 1)
    full_shifts = [(np.zeros(0, int), np.zeros(0, int))] * B
    for i, j in enumerate(r_np):
        full_shifts[j] = shifts[i]
    return out, tr, full_shifts


def run_audit(model, ctx, arr, marker_arr, has_marker, idx, fill_mode, seed, tag, clean_arr=None, n_examples=2):
    gen = torch.Generator().manual_seed(seed)
    lab, mok = META.label.values, META.mask_ok.values
    border = torch.zeros(IMG_SIZE, IMG_SIZE, dtype=torch.bool, device=DEVICE)
    border[:BORDER_PX] = True; border[-BORDER_PX:] = True
    border[:, :BORDER_PX] = True; border[:, -BORDER_PX:] = True
    keys = [f'{m}|{b}' for m in MAPS for b in BUDGETS] + [f'tumor|{b}' for b in BUDGETS]
    trials = {k: np.full((len(idx), N_RAND), np.nan, np.float32) for k in keys}
    rows, examples = [], []
    ex_left = {c: n_examples for c in range(N_CLASSES)}
    model.eval()
    for s in tqdm(range(0, len(idx), AUDIT_BATCH), desc=f'Audit {tag}', leave=False):
        gi = idx[s:s + AUDIT_BATCH]
        B, pos = len(gi), np.arange(s, s + len(gi))
        ar = torch.arange(B, device=DEVICE)
        x = to_batch(arr, gi)
        y = torch.as_tensor(lab[gi], device=DEVICE)
        has_t = torch.as_tensor(mok[gi], device=DEVICE)
        Tm = torch.from_numpy(MASKS[gi]).to(DEVICE) & has_t[:, None, None]
        FGm = torch.from_numpy(FG[gi]).to(DEVICE)
        MK = torch.from_numpy(marker_arr[gi]).to(DEVICE) if marker_arr is not None else torch.zeros_like(Tm)
        with torch.no_grad():
            with amp_ctx():
                logits0, _, A = model(x, return_maps=True)
            A = A.float()
            M0, L0 = margins(logits0, y)
            pred0 = logits0.argmax(1)
            maps = {m: class_map(m, A, y, ctx) for m in MAPS if m != 'ig'}
        if 'ig' in MAPS:
            maps['ig'] = ig_map(model, x, y)
        peri = dilate(Tm, PERI_RING_PX) & ~Tm
        head_o = FGm & ~Tm & ~peri
        bgd = ~FGm & ~Tm & ~peri
        t_area = Tm.float().sum((1, 2))
        mk_area = MK.float().sum((1, 2)).clamp_min(1)
        MKZ = dilate(MK, MARKER_ZONE_PX) if marker_arr is not None else MK
        correct0 = (pred0 == y).cpu().numpy()
        M0n, L0n, p0n = M0.cpu().numpy(), L0.cpu().numpy(), pred0.cpu().numpy()
        # Background arm (map-free): every pixel outside the head (foreground dilated by 5 px) set to black
        BGm = ~dilate(FGm, 5)
        lgb = forward_ablated(model, x, BGm, ar, 'black')
        mb_, _ = margins(lgb, y)
        dM_bg = (M0 - mb_).cpu().numpy()
        flip_bg = (lgb.argmax(1) != pred0).float().cpu().numpy()
        bg_frac = BGm.float().mean((1, 2)).cpu().numpy()
        # Background arm on the same image WITHOUT the marker (synthetic audits only)
        dM_bg_u = M0u_n = np.full(B, np.nan)
        if clean_arr is not None:
            with torch.no_grad():
                xu = to_batch(clean_arr, gi)
                with amp_ctx():
                    lu = model(xu)
                M0u, _ = margins(lu.float(), y)
                lgu = forward_ablated(model, xu, BGm, ar, 'black')
                mbu, _ = margins(lgu, y)
                dM_bg_u, M0u_n = (M0u - mbu).cpu().numpy(), M0u.cpu().numpy()
        # Oracle arm (marked data only): occlude exactly the marker → ground truth of marker use
        nan_b = np.full(B, np.nan)
        rmk = {'dM': nan_b, 'dM_rand_mean': nan_b, 'flip': nan_b, 'flip_rand': nan_b, 'n_rand': np.zeros(B)}
        if marker_arr is not None:
            has_mk = torch.as_tensor(has_marker[gi], device=DEVICE) & MK.flatten(1).any(1)
            rmk, _, _ = eval_region_and_null(model, x, y, M0, L0, pred0, MK, has_mk, Tm, FGm, fill_mode, gen)
        for b in BUDGETS:
            ref = float(round(b * REF_AREA))
            areas = torch.where(has_t, torch.round(b * t_area), torch.full_like(t_area, ref))
            areas = areas.clamp(1, IMG_SIZE * IMG_SIZE // 2).long()
            T3 = torch.from_numpy(tumor_region(gi, b)).to(DEVICE)
            r3, tr3, _ = eval_region_and_null(model, x, y, M0, L0, pred0, T3, has_t, Tm, FGm, fill_mode, gen)
            trials[f'tumor|{b}'][pos] = tr3
            for mname in MAPS:
                Mp = maps[mname]
                R = topk_mask(Mp, areas, gen)
                r1, tr1, shifts = eval_region_and_null(model, x, y, M0, L0, pred0, R,
                                                       torch.ones(B, dtype=torch.bool, device=DEVICE),
                                                       Tm, FGm, fill_mode, gen)
                trials[f'{mname}|{b}'][pos] = tr1
                a = R.float().sum((1, 2)).clamp_min(1)

                def frac(Z):
                    return ((R & Z).float().sum((1, 2)) / a).cpu().numpy()

                f_t, f_p, f_ho, f_bg = frac(Tm), frac(peri), frac(head_o), frac(bgd)
                f_bd, f_mk, f_fg = frac(border.expand_as(R)), frac(MK), frac(FGm)
                f_mkz = frac(MKZ)
                mk_rec = ((R & MK).float().sum((1, 2)) / mk_area).cpu().numpy()
                am = Mp.reshape(B, -1).argmax(1)
                py, px = am // IMG_SIZE, am % IMG_SIZE
                pt_t, pt_mk = Tm[ar, py, px].cpu().numpy(), MK[ar, py, px].cpu().numpy()
                posm = Mp.clamp_min(0)
                E = posm.sum((1, 2)).clamp_min(1e-12)
                en_t = ((posm * Tm).sum((1, 2)) / E).cpu().numpy()
                en_mk = ((posm * MK).sum((1, 2)) / E).cpu().numpy()
                ar_np = areas.cpu().numpy()
                for j in range(B):
                    g = gi[j]
                    rows.append(dict(
                        pos=pos[j], gidx=g, name=META.name.values[g], label=lab[g], cls=CLASS_NAMES[lab[g]],
                        plane=META.plane.values[g], dup=bool(META.dup.values[g]),
                        correct0=bool(correct0[j]), pred0=int(p0n[j]), margin0=M0n[j], logit0=L0n[j],
                        has_mask=bool(mok[g]), has_marker=bool(has_marker[g]) if has_marker is not None else False,
                        tumor_area=float(t_area[j]), budget=b, area=int(ar_np[j]), method=mname,
                        dM_art=r1['dM'][j], dL_art=r1['dL'][j], flip_art=r1['flip'][j],
                        dM_rand_mean=r1['dM_rand_mean'][j], dM_rand_std=r1['dM_rand_std'][j],
                        dL_rand_mean=r1['dL_rand_mean'][j], flip_rand=r1['flip_rand'][j],
                        n_rand_valid=int(r1['n_rand'][j]), p_emp_art=r1['p_emp'][j],
                        dM_tum=r3['dM'][j], dL_tum=r3['dL'][j], flip_tum=r3['flip'][j],
                        dM_rand_tum_mean=r3['dM_rand_mean'][j], dL_rand_tum_mean=r3['dL_rand_mean'][j],
                        flip_rand_tum=r3['flip_rand'][j], n_rand_tum_valid=int(r3['n_rand'][j]),
                        p_emp_tum=r3['p_emp'][j],
                        frac_tumor=f_t[j], frac_peri=f_p[j], frac_other_head=f_ho[j], frac_bg=f_bg[j],
                        frac_border=f_bd[j], frac_marker=f_mk[j], marker_recall=mk_rec[j],
                        frac_marker_zone=f_mkz[j], fg_frac_art=f_fg[j],
                        point_tumor=bool(pt_t[j]), energy_tumor=en_t[j],
                        point_marker=bool(pt_mk[j]), energy_marker=en_mk[j],
                        dM_marker=rmk['dM'][j], dM_rand_marker_mean=rmk['dM_rand_mean'][j],
                        flip_marker=rmk['flip'][j], flip_rand_marker=rmk['flip_rand'][j],
                        n_rand_marker_valid=int(rmk['n_rand'][j]),
                        dM_bg=dM_bg[j], flip_bg=flip_bg[j], bg_frac=bg_frac[j],
                        dM_bg_unmarked=dM_bg_u[j], margin0_unmarked=M0u_n[j]))
                if mname == MAPS[0] and b == PRIMARY_BUDGET:
                    for j in range(B):
                        c = int(lab[gi[j]])
                        if ex_left[c] > 0 and correct0[j]:
                            ex_left[c] -= 1
                            dy, dx = shifts[j]
                            rnd = [torch.roll(R[j], (int(u), int(v)), (0, 1)).cpu().numpy() for u, v in zip(dy[:3], dx[:3])]
                            examples.append(dict(img=arr[gi[j]].copy(), tumor=Tm[j].cpu().numpy(), marker=MK[j].cpu().numpy(),
                                                 maps={m: maps[m][j].cpu().numpy() for m in MAPS},
                                                 region=R[j].cpu().numpy(), rand=rnd, label=c,
                                                 dM_art=r1['dM'][j], dM_rand=r1['dM_rand_mean'][j], dM_tum=r3['dM'][j]))
    return pd.DataFrame(rows), trials, examples

## 8. Statistics and verdict rules

Primary metric: drop of the margin (true-class logit minus the largest other logit). Wilcoxon signed-rank tests with
rank-biserial r, Benjamini–Hochberg correction within each subset (`all`, `dedup`, `marker`). Effects are expressed
relative to the median baseline margin of the class.

| Verdict | Rule (significant = q < 0.05, r ≥ 0.30, effect ≥ 5% of the median margin) |
|---|---|
| OFF-TUMOR RELIANCE [location] | Arm 1 > null, Arm 1 > Arm 3, < 25% of the Arm-1 region in the tumour |
| MIXED: TUMOR + CONTEXT [location] | Arm 1 > null and Arm 3 > null, Arm-1 region outside the tumour, Arm 1 not > Arm 3 |
| PATHOLOGY-GROUNDED | Arm 3 > tumour null (otherwise) |
| REGION-SPECIFIC [location] | Arm 1 > null without tumour-specific evidence (including no_tumour) |
| INERT / DISTRIBUTED | nothing significant and flip rates of all arms < 2% |
| NON-SPECIFIC (OOD-SENSITIVE) | nothing significant, random translations flip ≥ 10% of predictions |
| INCONCLUSIVE | otherwise |

Location: `peritumoral` (15 px ring), `other-head` (head outside tumour and ring, including skull, orbits and
sinuses) or `background`; `+border` is added when ≥ 50% of the region lies in the 16 px image border.

In [ ]:
# ===================================================================
# 8) STATISTICS & VERDICTS
# ===================================================================
def wilcoxon_test(d, alternative):
    d = np.asarray(d, float)
    d = d[np.isfinite(d)]
    nz = d[d != 0]
    if len(nz) < 6:
        return np.nan, np.nan
    p = stats.wilcoxon(nz, alternative=alternative).pvalue
    rk = stats.rankdata(np.abs(nz))
    wp, wm = rk[nz > 0].sum(), rk[nz < 0].sum()
    return float(p), float((wp - wm) / (wp + wm))


def mean_ci(v, seed=0):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return np.nan, np.nan, np.nan
    if len(v) == 1:
        return float(v[0]), np.nan, np.nan
    bs = v[np.random.default_rng(seed).integers(0, len(v), (N_BOOT, len(v)))].mean(1)
    return float(v.mean()), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))


def null_means(tr, n_draw, seed=0):
    tr = np.asarray(tr, float)
    cnt = np.isfinite(tr).sum(1)
    tr, cnt = tr[cnt > 0], cnt[cnt > 0]
    if len(tr) == 0:
        return np.array([])
    tr = np.take_along_axis(tr, np.argsort(~np.isfinite(tr), axis=1, kind='stable'), 1)
    pick = (np.random.default_rng(seed).random((n_draw, len(tr))) * cnt[None, :]).astype(int)
    return tr[np.arange(len(tr))[None, :], pick].mean(1)


def med_pos(v):
    v = np.asarray(v, float)
    v = v[np.isfinite(v) & (v > 0)]
    return float(np.median(v)) if len(v) else np.nan


def p_upper(null, obs):
    if len(null) == 0 or not np.isfinite(obs):
        return np.nan
    return float((1 + (null >= obs).sum()) / (len(null) + 1))


def location_tag(r):
    parts = {'peritumoral': r['frac_peri'], 'other-head': r['frac_other_head'], 'background': r['frac_bg']}
    tag = max(parts, key=lambda k: parts[k] if np.isfinite(parts[k]) else -1)
    if np.isfinite(r['frac_border']) and r['frac_border'] >= 0.5:
        tag += '+border'
    return tag


def verdict_of(r):
    if r['n_valid'] < 6:
        return 'INSUFFICIENT DATA'
    has_ref = r['n_tum'] >= 6
    sig_art = (r['q_art'] < ALPHA) and (r['r_art'] >= R_MIN) and (r['spec_art_rel'] >= REL_MIN)
    sig_tum = has_ref and (r['q_tum'] < ALPHA) and (r['r_tum'] >= R_MIN) and (r['spec_tum_rel'] >= REL_MIN)
    art_gt_tum = has_ref and (r['q_art_tum'] < ALPHA) and (r['r_art_tum'] >= R_MIN) and (r['art_minus_tum_rel'] >= REL_MIN)
    off = r['frac_tumor'] < IN_TUMOR_MAX
    flips = [r['flip_art'], r['flip_rand']] + ([r['flip_tum'], r['flip_rand_tum']] if has_ref else [])
    inert = all(np.isfinite(f) and f < INERT_FLIP for f in flips)
    if sig_art and art_gt_tum and off:
        return f'OFF-TUMOR RELIANCE [{location_tag(r)}]'
    if sig_art and sig_tum and off:
        return f'MIXED: TUMOR + CONTEXT [{location_tag(r)}]'
    if sig_tum:
        return 'PATHOLOGY-GROUNDED'
    if sig_art:
        return 'REGION-SPECIFIC [' + (location_tag(r) if (off or not has_ref) else 'on-tumor') + ']'
    if inert:
        return 'INERT / DISTRIBUTED'
    if np.isfinite(r['flip_rand']) and r['flip_rand'] >= OOD_FLIP:
        return 'NON-SPECIFIC (OOD-SENSITIVE)'
    return 'INCONCLUSIVE'


SPATIAL_COLS = ['frac_tumor', 'frac_peri', 'frac_other_head', 'frac_bg', 'frac_border', 'frac_marker',
                'marker_recall', 'frac_marker_zone', 'fg_frac_art', 'point_tumor', 'energy_tumor', 'point_marker',
                'energy_marker']


def summarize_audit(df, trials, subsets):
    X, out = PRIMARY, []
    for (mname, b), d0 in df.groupby(['method', 'budget'], sort=False):
        tr_map, tr_tum = trials[f'{mname}|{b}'], trials[f'tumor|{b}']
        for sub_name, sub_fn in subsets:
            for c in range(N_CLASSES):
                d = d0[(d0.label.values == c) & sub_fn(d0)]
                if ONLY_CORRECT:
                    d = d[d.correct0.astype(bool)]
                dv = d[d.n_rand_valid >= MIN_VALID_TRIALS]
                dt = d[d.has_mask.astype(bool) & (d.n_rand_tum_valid >= MIN_VALID_TRIALS)]
                r = dict(method=mname, budget=b, subset=sub_name, label=c, cls=CLASS_NAMES[c], n_img=len(d),
                         n_valid=len(dv), n_rand_fail=len(d) - len(dv), n_tum=len(dt),
                         n_tum_missing=int((~d.has_mask.astype(bool)).sum()) if c != 0 else 0)
                for arm, src, col in [('art', dv, f'{X}_art'), ('rand', dv, f'{X}_rand_mean'),
                                      ('tum', dt, f'{X}_tum'), ('rand_tum', dt, f'{X}_rand_tum_mean')]:
                    r[f'{arm}_mean'], r[f'{arm}_lo'], r[f'{arm}_hi'] = mean_ci(src[col].values)
                m0v, m0t = med_pos(dv.margin0.values), med_pos(dt.margin0.values)
                d_art = dv[f'{X}_art'].values - dv[f'{X}_rand_mean'].values
                r['spec_art'] = float(np.mean(d_art)) if len(d_art) else np.nan
                r['spec_art_rel'] = r['spec_art'] / m0v
                r['p_art'], r['r_art'] = wilcoxon_test(d_art, 'greater')
                r['p_art_null'] = p_upper(null_means(tr_map[dv.pos.values], N_NULL),
                                          dv[f'{X}_art'].mean() if len(dv) else np.nan)
                if len(dt) >= 6:
                    d_tum = dt[f'{X}_tum'].values - dt[f'{X}_rand_tum_mean'].values
                    r['spec_tum'] = float(np.mean(d_tum))
                    r['spec_tum_rel'] = r['spec_tum'] / m0t
                    r['p_tum'], r['r_tum'] = wilcoxon_test(d_tum, 'greater')
                    r['p_tum_null'] = p_upper(null_means(tr_tum[dt.pos.values], N_NULL), dt[f'{X}_tum'].mean())
                    d_at = dt[f'{X}_art'].values - dt[f'{X}_tum'].values
                    r['art_minus_tum'] = float(np.mean(d_at))
                    r['art_minus_tum_rel'] = r['art_minus_tum'] / m0t
                    r['p_art_tum'], r['r_art_tum'] = wilcoxon_test(d_at, 'two-sided')
                else:
                    for k in ['spec_tum', 'spec_tum_rel', 'p_tum', 'r_tum', 'p_tum_null', 'art_minus_tum',
                              'art_minus_tum_rel', 'p_art_tum', 'r_art_tum']:
                        r[k] = np.nan
                r['margin0_median'] = float(np.median(d.margin0.values)) if len(d) else np.nan
                r['flip_art'] = float(dv.flip_art.mean()) if len(dv) else np.nan
                r['flip_rand'] = float(dv.flip_rand.mean()) if len(dv) else np.nan
                r['flip_tum'] = float(dt.flip_tum.mean()) if len(dt) else np.nan
                r['flip_rand_tum'] = float(dt.flip_rand_tum.mean()) if len(dt) else np.nan
                r['dL_art_mean'] = float(dv.dL_art.mean()) if len(dv) else np.nan
                r['dL_rand_mean'] = float(dv.dL_rand_mean.mean()) if len(dv) else np.nan
                r['dL_tum_mean'] = float(dt.dL_tum.mean()) if len(dt) else np.nan
                r['frac_img_p05_art'] = float((dv.p_emp_art < 0.05).mean()) if len(dv) else np.nan
                for col in SPATIAL_COLS:
                    r[col] = float(d[col].astype(float).mean()) if (len(d) and col in d) else np.nan
                # background arm (map-free)
                if len(d) >= 6:
                    r['bg_mean'] = float(d.dM_bg.mean())
                    r['bg_rel'] = r['bg_mean'] / med_pos(d.margin0.values)
                    r['p_bg'], r['r_bg'] = wilcoxon_test(d.dM_bg.values, 'greater')
                    r['flip_bg'] = float(d.flip_bg.mean())
                    r['bg_frac'] = float(d.bg_frac.mean())
                else:
                    r['bg_mean'] = r['bg_rel'] = r['p_bg'] = r['r_bg'] = r['flip_bg'] = r['bg_frac'] = np.nan
                # background reliance CAUSED by the marker (synthetic audits)
                du = d[np.isfinite(d.dM_bg_unmarked.astype(float))] if 'dM_bg_unmarked' in d else d.iloc[:0]
                if len(du) >= 6:
                    d_bgm = du.dM_bg.values - du.dM_bg_unmarked.values
                    r['bg_marker_rel'] = float(np.mean(d_bgm)) / med_pos(du.margin0.values)
                    r['p_bg_marker'], r['r_bg_marker'] = wilcoxon_test(d_bgm, 'greater')
                else:
                    r['bg_marker_rel'] = r['p_bg_marker'] = r['r_bg_marker'] = np.nan
                # ground truth (synthetic data): oracle arm vs its translations
                dk = d[d.has_marker.astype(bool) & (d.n_rand_marker_valid >= MIN_VALID_TRIALS)]
                r['n_marker'] = len(dk)
                if len(dk) >= 6:
                    d_mk = dk.dM_marker.values - dk.dM_rand_marker_mean.values
                    r['oracle_marker_rel'] = float(np.mean(d_mk)) / med_pos(dk.margin0.values)
                    r['p_marker'], r['r_marker'] = wilcoxon_test(d_mk, 'greater')
                    r['flip_marker'] = float(dk.flip_marker.mean())
                else:
                    r['oracle_marker_rel'] = r['p_marker'] = r['r_marker'] = r['flip_marker'] = np.nan
                out.append(r)
    S = pd.DataFrame(out)
    for col in ['q_art', 'q_tum', 'q_art_tum', 'q_marker']:
        S[col] = np.nan
    for _, ix in S.groupby('subset').groups.items():       # BH correction within each subset
        ix = list(ix)
        n = len(ix)
        q = bh(np.concatenate([S.loc[ix, 'p_art'].values, S.loc[ix, 'p_tum'].values, S.loc[ix, 'p_art_tum'].values]))
        S.loc[ix, 'q_art'], S.loc[ix, 'q_tum'], S.loc[ix, 'q_art_tum'] = q[:n], q[n:2 * n], q[2 * n:]
        S.loc[ix, 'q_marker'] = bh(S.loc[ix, 'p_marker'].values)
    S['verdict'] = [verdict_of(r) for r in S.to_dict('records')]
    S['tumor_grounded'] = (S.q_tum < ALPHA) & (S.r_tum >= R_MIN) & (S.spec_tum_rel >= REL_MIN)
    S['oracle_marker_reliance'] = (S.q_marker < ALPHA) & (S.r_marker >= R_MIN) & (S.oracle_marker_rel >= REL_MIN)
    # the background arm does not depend on map/budget → one BH correction per (subset, class)
    first = S.drop_duplicates(['subset', 'label'])[['subset', 'label', 'p_bg', 'p_bg_marker']].copy()
    first['q_bg'], first['q_bg_marker'] = np.nan, np.nan
    for _, ix in first.groupby('subset').groups.items():
        ix = list(ix)
        first.loc[ix, 'q_bg'] = bh(first.loc[ix, 'p_bg'].values)
        first.loc[ix, 'q_bg_marker'] = bh(first.loc[ix, 'p_bg_marker'].values)
    S = S.merge(first[['subset', 'label', 'q_bg', 'q_bg_marker']], on=['subset', 'label'], how='left')
    S['background_reliance'] = (S.q_bg < ALPHA) & (S.r_bg >= R_MIN) & (S.bg_rel >= REL_MIN)
    S['background_marker_reliance'] = (S.q_bg_marker < ALPHA) & (S.r_bg_marker >= R_MIN) & (S.bg_marker_rel >= REL_MIN)
    return S


def print_verdicts(S, subset, budget=None, maps=None, classes=None):
    budget = PRIMARY_BUDGET if budget is None else budget
    d = S[(S.subset == subset) & np.isclose(S.budget, budget) & (S['method'].isin(maps or MAPS))
          & S.cls.isin(classes or CLASS_NAMES)]
    cols = ['method', 'cls', 'n_valid', 'art_mean', 'rand_mean', 'tum_mean', 'rand_tum_mean', 'spec_art_rel',
            'r_art', 'q_art', 'spec_tum_rel', 'r_tum', 'q_tum', 'flip_rand', 'frac_tumor', 'verdict']
    with pd.option_context('display.width', 250, 'display.max_columns', 30, 'display.max_colwidth', 45):
        print(d[cols].to_string(index=False, float_format=lambda v: f'{v:.3g}'))

## 9. Per-audit figures

In [ ]:
# ===================================================================
# 9) FIGURES
# ===================================================================
def plot_null(df, S, trials, mname, b, title, path, show=False, subset='all'):
    fig, axes = plt.subplots(1, N_CLASSES, figsize=(2.9 * N_CLASSES, 2.7))
    lbl = 'Mean margin drop' if PRIMARY == 'dM' else 'Mean logit drop'
    for c, ax in enumerate(axes):
        d = df[(df['method'] == mname) & np.isclose(df.budget, b) & (df.label == c)]
        if subset == 'marker':
            d = d[d.has_marker.astype(bool)]
        elif subset == 'dedup':
            d = d[~d.dup.astype(bool)]
        if ONLY_CORRECT:
            d = d[d.correct0.astype(bool)]
        dv = d[d.n_rand_valid >= MIN_VALID_TRIALS]
        if len(dv) == 0:
            ax.set_axis_off()
            continue
        null = null_means(trials[f'{mname}|{b}'][dv.pos.values], N_NULL)
        ax.hist(null, bins=40, density=True, color='0.7', label='Null (map-region translations)')
        ax.axvline(dv[f'{PRIMARY}_art'].mean(), color='tab:red', ls='--', lw=1.5, label='Arm 1: map region')
        dt = d[d.has_mask.astype(bool) & (d.n_rand_tum_valid >= MIN_VALID_TRIALS)]
        if len(dt):
            null_t = null_means(trials[f'tumor|{b}'][dt.pos.values], N_NULL)
            ax.hist(null_t, bins=40, density=True, color='tab:green', alpha=0.3, label='Null (tumour translations)')
            ax.axvline(dt[f'{PRIMARY}_tum'].mean(), color='tab:green', lw=1.5, label='Arm 3: tumour')
        row = S[(S['method'] == mname) & np.isclose(S.budget, b) & (S.label == c) & (S.subset == subset)]
        ax.set_title(f'{CLASS_NAMES[c]} (n={len(dv)})\n{row.verdict.iloc[0] if len(row) else ""}', fontsize=7)
        ax.set_xlabel(lbl)
    handles = {}
    for ax in axes:
        for h_, l_ in zip(*ax.get_legend_handles_labels()):
            handles.setdefault(l_, h_)
    fig.legend(list(handles.values()), list(handles.keys()), loc='lower center', ncol=4, fontsize=6, frameon=False)
    fig.suptitle(title, fontsize=9)
    fig.tight_layout(rect=(0, 0.08, 1, 0.94))
    save_fig(fig, path, show)


def plot_arms(S, b, title, path, show=False, subset='all'):
    d = S[np.isclose(S.budget, b) & (S.subset == subset)]
    fig, axes = plt.subplots(1, len(MAPS), figsize=(3.0 * len(MAPS), 2.8), sharey=True)
    axes = np.atleast_1d(axes)
    arms = [('art', 'Arm 1: map', 'tab:red'), ('rand', 'Null of Arm 1', '0.6'),
            ('tum', 'Arm 3: tumour', 'tab:green'), ('rand_tum', 'Null of Arm 3', '0.85')]
    xs = np.arange(N_CLASSES)
    for ax, mname in zip(axes, MAPS):
        dm = d[d['method'] == mname].set_index('label').reindex(range(N_CLASSES))
        for i, (arm, lab_, col) in enumerate(arms):
            m_ = np.nan_to_num(dm[f'{arm}_mean'].values)
            err = np.vstack([m_ - np.nan_to_num(dm[f'{arm}_lo'].values, nan=0.0),
                             np.nan_to_num(dm[f'{arm}_hi'].values, nan=0.0) - m_])
            ax.bar(xs + (i - 1.5) * 0.2, m_, 0.2, color=col, label=lab_,
                   yerr=np.clip(np.nan_to_num(err), 0, None), capsize=1.5, error_kw={'lw': 0.6})
        ax.axhline(0, color='k', lw=0.5)
        ax.set_xticks(xs)
        ax.set_xticklabels([n.replace('_', ' ') for n in CLASS_NAMES], fontsize=7)
        ax.set_title(mname)
    axes[0].set_ylabel(('Margin' if PRIMARY == 'dM' else 'Logit') + ' drop (mean, 95% CI)')
    axes[0].legend(fontsize=6)
    fig.suptitle(title, fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_fig(fig, path, show)


def plot_location(S, b, title, path, show=False, subset='all'):
    d = S[np.isclose(S.budget, b) & (S.subset == subset)]
    parts = [('frac_tumor', 'tumour', 'tab:green'), ('frac_peri', 'peritumoural', 'yellowgreen'),
             ('frac_other_head', 'other head', 'tab:blue'), ('frac_bg', 'background', '0.5')]
    fig, axes = plt.subplots(1, len(MAPS), figsize=(2.8 * len(MAPS), 2.6), sharey=True)
    axes = np.atleast_1d(axes)
    for ax, mname in zip(axes, MAPS):
        dm = d[d['method'] == mname].set_index('label').reindex(range(N_CLASSES))
        bottom = np.zeros(N_CLASSES)
        for col, lab_, color in parts:
            v = np.nan_to_num(dm[col].values)
            ax.bar(range(N_CLASSES), v, bottom=bottom, color=color, label=lab_)
            bottom += v
        ax.set_xticks(range(N_CLASSES))
        ax.set_xticklabels([n.replace('_', ' ') for n in CLASS_NAMES], fontsize=7)
        ax.set_title(mname)
    axes[0].set_ylabel('Composition of Arm-1 region')
    axes[0].legend(fontsize=6)
    fig.suptitle(title, fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_fig(fig, path, show)


def plot_examples(examples, title, path, show=False):
    if not examples:
        return
    ncol = 1 + len(MAPS) + 1
    fig, axes = plt.subplots(len(examples), ncol, figsize=(1.8 * ncol, 1.8 * len(examples)))
    axes = np.atleast_2d(axes)
    for r, ex in enumerate(examples):
        ax = axes[r, 0]
        ax.imshow(ex['img'], cmap='gray')
        if ex['tumor'].any():
            ax.contour(ex['tumor'], levels=[0.5], colors='lime', linewidths=0.7)
        if ex['marker'].any():
            ax.contour(ex['marker'], levels=[0.5], colors='magenta', linewidths=0.7)
        ax.set_ylabel(CLASS_NAMES[ex['label']], fontsize=7)
        for k, mname in enumerate(MAPS):
            ax = axes[r, 1 + k]
            mp = np.clip(ex['maps'][mname], 0, None)
            ax.imshow(ex['img'], cmap='gray')
            ax.imshow(mp / (mp.max() + 1e-12), cmap='jet', alpha=0.45)
            if r == 0:
                ax.set_title(mname, fontsize=7)
        ax = axes[r, -1]
        ax.imshow(ex['img'], cmap='gray')
        ax.contour(ex['region'], levels=[0.5], colors='red', linewidths=0.8)
        for rm in ex['rand']:
            ax.contour(rm, levels=[0.5], colors='cyan', linewidths=0.5)
        if ex['tumor'].any():
            ax.contour(ex['tumor'], levels=[0.5], colors='lime', linewidths=0.5)
        ax.set_xlabel(f"Δ art {ex['dM_art']:.2f} | rand {ex['dM_rand']:.2f} | tum {ex['dM_tum']:.2f}", fontsize=5.5)
        if r == 0:
            ax.set_title(f'Arm 1 ({MAPS[0]}, {PRIMARY_BUDGET:g}×) + null', fontsize=7)
        for a in axes[r]:
            a.set_xticks([]); a.set_yticks([])
    fig.suptitle(title, fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    save_fig(fig, path, show)

## 10. Run orchestration

Priority order (so that the core results exist first if a session stops):
1. Clean ResNet50 seed 0: real-data audit, fill sensitivity, and negative controls for the three synthetic conditions.
2. Clean ResNet50 seeds 1 and 2: real-data audit and negative controls.
3. Shortcut models: 3 conditions × 3 seeds.
4. Clean DenseNet121 and EfficientNet-B0 (3 seeds each).

In [ ]:
# ===================================================================
# 10) ORCHESTRATION
# ===================================================================
RUN_RE = re.compile(r'^(clean|shortcut-([A-Za-z0-9]+))_(.+)_s(\d+)$')


def run_name(kind, cond, arch, seed):
    return f'clean_{arch}_s{seed}' if kind == 'clean' else f'shortcut-{cond}_{arch}_s{seed}'


def run_dir_of(name):
    d = os.path.join(OUT_DIR, 'runs', name)
    os.makedirs(os.path.join(d, 'figures'), exist_ok=True)
    return d


def get_model_and_context(kind, cond, arch, seed, show=False):
    rd = run_dir_of(run_name(kind, cond, arch, seed))
    arr = IMGS if kind == 'clean' else SYN[cond]['imgs']
    model = train_model(arch, seed, arr, IDX_TR, IDX_VA, rd)
    ytr, yte = META.label.values[IDX_TR], META.label.values[IDX_TE]
    f_spec = os.path.join(rd, 'spectral.npz')
    if os.path.exists(f_spec) and not FORCE_RERUN and 'mu_sp' in np.load(f_spec).files \
            and os.path.exists(os.path.join(rd, 'test_metrics.json')):
        z = np.load(f_spec)
        spec = {k: z[k] for k in ['mu', 'V', 'eig']}
        spec.update(load_json(os.path.join(rd, 'spectral_summary.json'))['spectrum'])
        W, sel, Wd, K = z['W'], z['sel'], z['Wd'], int(z['K'])
        print(f'  spectral analysis loaded (K_MP = {spec["K_mp"]})')
        return model, make_map_ctx(spec, K, sel, Wd, W, z['mu_sp']), rd
    Lte = predict(model, arr, IDX_TE)
    rep = classification_report(Lte, yte)
    nd = ~META.dup.values[IDX_TE]
    rep['acc_dedup'] = float((Lte.argmax(1)[nd] == yte[nd]).mean()) if nd.any() else np.nan
    rep['bal_acc_dedup'] = float(balanced_accuracy_score(yte[nd], Lte.argmax(1)[nd])) if nd.any() else np.nan
    rep['n_dup_removed'] = int((~nd).sum())
    save_json(rep, os.path.join(rd, 'test_metrics.json'))
    pd.DataFrame({'name': META.name.values[IDX_TE], 'label': yte, 'pred': Lte.argmax(1),
                  'dup': ~nd}).to_csv(os.path.join(rd, 'test_predictions.csv'), index=False)
    print(f'  test ({"marked, in-distribution" if kind != "clean" else "clean"}): acc {rep["acc"]:.4f} | '
          f'bal_acc {rep["bal_acc"]:.4f} | macro-F1 {rep["macro_f1"]:.4f} | acc dedup {rep["acc_dedup"]:.4f}')
    t0 = time.time()
    Ltr, Ftr, mu_sp = predict(model, arr, IDX_TR, want_features=True, want_spatial_mean=True)
    Lte, Fte = predict(model, arr, IDX_TE, want_features=True)
    W = model.fc.weight.detach().double().cpu().numpy()
    bias = model.fc.bias.detach().double().cpu().numpy()
    err = np.abs(Lte - (Fte @ W.T + bias)).max()
    assert err < 1e-3 * max(1.0, np.abs(Lte).max()), f'logits ≠ W·pooled + b (err {err})'
    spec = spectral_fit(Ftr, seed)
    K = spec['K_mp'] if spec['K_mp'] > 0 else max(spec['K_pa'], 1)
    comp, sel, S_contrib, Wd, Ute, WdV = component_analysis(spec, K, W, Ftr, ytr, Fte, yte)
    fid = fidelity_analysis(Ftr, ytr, Fte, yte, W, bias, spec, K, seed)
    comp.to_csv(os.path.join(rd, 'components.csv'), index=False)
    np.savez(f_spec, mu=spec['mu'], V=spec['V'], eig=spec['eig'], pa_thr=spec['pa_thr'], W=W, b=bias, Wd=Wd,
             sel=sel, S=S_contrib, K=K, mu_sp=mu_sp)
    spectrum = {k: spec[k] for k in ['N', 'D', 'sigma2', 'sigma2_median', 'edge', 'edge_asym', 'q', 'K_mp',
                                     'K_mp_asym', 'K_mp_median', 'K_gd', 'K_pa', 'tau_gd_eig', 'pa_thr']}
    save_json({'spectrum': spectrum, 'fidelity': fid,
               'selected_component': {CLASS_NAMES[c]: int(sel[c]) for c in range(N_CLASSES)}},
              os.path.join(rd, 'spectral_summary.json'))
    print(f'  spectral: K_MP={spec["K_mp"]}  K_GD={spec["K_gd"]}  K_PA={spec["K_pa"]}  ({time.time() - t0:.0f}s)')
    tag = os.path.basename(rd)
    plot_spectrum(spec, f'Spectrum of GAP features — {tag}', os.path.join(rd, 'figures', 'spectrum.pdf'), show)
    plot_components(Ute, yte, sel, WdV, f'Selected components (test scores) — {tag}',
                    os.path.join(rd, 'figures', 'components.pdf'), False)
    return model, make_map_ctx(spec, K, sel, Wd, W, mu_sp), rd


def audit_job(model, ctx, rd, test_set, fill_mode, seed, show=False):
    name = f'audit_{test_set}_{fill_mode}'
    f_sum = os.path.join(rd, f'{name}_summary.csv')
    if os.path.exists(f_sum) and not FORCE_RERUN:
        print(f'  {name}: already done, skipped')
        return pd.read_csv(f_sum)
    if test_set == 'clean':
        arr, mk, hm, carr, idx = IMGS, None, None, None, IDX_TE
        subsets = [('dedup', lambda d: ~d.dup.astype(bool).values), ('all', lambda d: np.ones(len(d), bool))]
        main_sub = 'dedup'
    else:
        cond = test_set[len('syn-'):]
        arr, mk, hm, carr = SYN[cond]['imgs'], SYN[cond]['mark'], SYN[cond]['has'], IMGS
        idx = IDX_TE[META.label.values[IDX_TE] == TARGET_IDX] if SYN_AUDIT_TARGET_ONLY else IDX_TE
        subsets = [('marker', lambda d: d.has_marker.astype(bool).values), ('all', lambda d: np.ones(len(d), bool))]
        main_sub = 'marker'
    t0 = time.time()
    f_img, f_tr = os.path.join(rd, f'{name}_images.csv'), os.path.join(rd, f'{name}_trials.npz')
    if os.path.exists(f_img) and os.path.exists(f_tr) and not FORCE_RERUN:
        df = pd.read_csv(f_img)
        z = np.load(f_tr)
        trials, examples = {k.replace('__', '|'): z[k] for k in z.files}, []
        print(f'  {name}: using saved ablation results, recomputing statistics')
    else:
        df, trials, examples = run_audit(model, ctx, arr, mk, hm, idx, fill_mode, seed=1000 + seed,
                                         tag=f'{os.path.basename(rd)} {test_set}/{fill_mode}', clean_arr=carr)
        df.to_csv(f_img, index=False)
        np.savez_compressed(f_tr, **{k.replace('|', '__'): v for k, v in trials.items()})
    S = summarize_audit(df, trials, subsets)
    S.to_csv(f_sum, index=False)
    tag = f'{os.path.basename(rd)} | {test_set} | fill={fill_mode}'
    fdr = os.path.join(rd, 'figures')
    plot_budgets = BUDGETS if test_set == 'clean' else [PRIMARY_BUDGET]
    for b in plot_budgets:
        if test_set == 'clean':
            for mname in MAPS:
                plot_null(df, S, trials, mname, b, f'{tag} | {mname} | {b:g}× tumour area',
                          os.path.join(fdr, f'{name}_null_{mname}_b{b:g}.pdf'),
                          show and mname == MAPS[0] and b == PRIMARY_BUDGET, subset=main_sub)
            plot_location(S, b, f'{tag} | {b:g}× tumour area', os.path.join(fdr, f'{name}_location_b{b:g}.pdf'),
                          False, subset=main_sub)
        plot_arms(S, b, f'{tag} | {b:g}× tumour area', os.path.join(fdr, f'{name}_arms_b{b:g}.pdf'),
                  show and b == PRIMARY_BUDGET, subset=main_sub)
    plot_examples(examples, tag, os.path.join(fdr, f'{name}_examples.pdf'), show)
    print(f'  {name}: finished in {(time.time() - t0) / 60:.1f} min')
    print_verdicts(S, subset=main_sub, classes=None if test_set == 'clean' else [SYN_TARGET])
    bgt = S[(S.subset == main_sub) & (S['method'] == MAPS[0]) & np.isclose(S.budget, PRIMARY_BUDGET)]
    print('  Background arm (outside of head set to black, map-free):')
    with pd.option_context('display.width', 200):
        print(bgt[['cls', 'n_img', 'bg_rel', 'r_bg', 'q_bg', 'flip_bg', 'background_reliance']].to_string(
            index=False, float_format=lambda v: f'{v:.3g}'))
    if test_set != 'clean':
        o = S[(S.subset == 'marker') & (S.label == TARGET_IDX) & np.isclose(S.budget, PRIMARY_BUDGET)]
        if len(o):
            r0 = o.iloc[0]
            print(f'  ORACLE ({SYN_TARGET}, n={int(r0.n_marker)}): occluding the marker lowers the margin by '
                  f'{100 * r0.oracle_marker_rel:.1f}% (r={r0.r_marker:.2f}, q={r0.q_marker:.2g}, flip={r0.flip_marker:.2f})'
                  f' → model {"USES" if r0.oracle_marker_reliance else "shows no evidence of using"} the marker')
            print(f'  marker-induced background reliance: {100 * r0.bg_marker_rel:.1f}% of margin '
                  f'({"significant" if r0.background_marker_reliance else "not significant"})')
            print(f'  fraction of the Arm-1 region inside the marker zone (±{MARKER_ZONE_PX} px): '
                  + ', '.join(f'{m}={v:.2f}' for m, v in zip(o['method'], o.frac_marker_zone)))
    return S


def behavior_job(model, rd, cond):
    f = os.path.join(rd, f'behavior_{cond}.json')
    if os.path.exists(f) and not FORCE_RERUN:
        return load_json(f)
    y = META.label.values[IDX_TE]
    out = {}
    for name, arr in [('clean', IMGS), ('synthetic_ID', SYN[cond]['imgs']), ('all_marked', SYN[cond]['imgs_all'])]:
        pred = predict(model, arr, IDX_TE).argmax(1)
        out[name] = {'acc': float((pred == y).mean()),
                     'recall': {CLASS_NAMES[c]: float((pred[y == c] == c).mean()) if (y == c).any() else float('nan')
                                for c in range(N_CLASSES)},
                     'pred_as_target': {CLASS_NAMES[c]: float((pred[y == c] == TARGET_IDX).mean()) if (y == c).any()
                                        else float('nan') for c in range(N_CLASSES)}}
    save_json(out, f)
    print(f'  behaviour [{cond}]: ' + ' | '.join(f'{k}: acc {v["acc"]:.3f}' for k, v in out.items())
          + f' | predicted {SYN_TARGET} when every image is marked: '
          + ', '.join(f'{c}={v:.2f}' for c, v in out['all_marked']['pred_as_target'].items() if c != SYN_TARGET))
    return out


# Run plan: (kind, cond, arch, seed, audits, behaviour conditions)
PLAN, _added = [], set()
FIRST = (ARCHS[0], SEEDS[0])


def _add_clean(a, s):
    if (a, s) in _added:
        return
    _added.add((a, s))
    audits = [('clean', FILL_MODE)]
    if (a, s) == FIRST:
        audits += [('clean', f) for f in SENSITIVITY_FILLS if f != FILL_MODE]
    behs = []
    if RUN_SYNTHETIC and a == SYN_ARCH and s in SYN_SEEDS:
        audits += [(f'syn-{c}', FILL_MODE) for c in SYN_CONDS]
        behs = list(SYN_CONDS)
    PLAN.append(('clean', None, a, s, audits, behs))


_add_clean(*FIRST)
if RUN_SYNTHETIC:
    for _s in SYN_SEEDS:
        _add_clean(SYN_ARCH, _s)
    for _c in SYN_CONDS:
        for _s in SYN_SEEDS:
            PLAN.append(('shortcut', _c, SYN_ARCH, _s, [(f'syn-{_c}', FILL_MODE)], [_c]))
for _a in ARCHS:
    for _s in SEEDS:
        _add_clean(_a, _s)
print(f'{len(PLAN)} model runs:')
for _p in PLAN:
    print(f'  {run_name(_p[0], _p[1], _p[2], _p[3]):34s} → audit {[f"{t}/{f}" for t, f in _p[4]]}'
          + (f' + behaviour {_p[5]}' if _p[5] else ''))

In [ ]:
# ===================================================================
# 11) EXECUTION
# ===================================================================
_shown_conds = set()
for i, (kind, cond, arch, seed, audits, behs) in enumerate(PLAN):
    if hours_used() > MAX_HOURS:
        msg = f'Time limit of {MAX_HOURS} h reached: skipping the remaining {len(PLAN) - i} runs.'
        print(msg)
        with open(os.path.join(OUT_DIR, 'errors.log'), 'a') as f:
            f.write(f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] {msg}\n')
        break
    name = run_name(kind, cond, arch, seed)
    print(f'\n{"=" * 90}\n[{i + 1}/{len(PLAN)}] {name} | elapsed {hours_used() * 60:.0f} min\n{"=" * 90}')
    model = ctx = None
    try:
        show = (i == 0) or (kind == 'shortcut' and cond not in _shown_conds)
        if kind == 'shortcut':
            _shown_conds.add(cond)
        model, ctx, rd = get_model_and_context(kind, cond, arch, seed, show=(i == 0))
        for ts, fm in audits:
            if hours_used() > MAX_HOURS:
                print('  time limit reached, skipping the remaining audits')
                break
            audit_job(model, ctx, rd, ts, fm, seed, show=show and fm == FILL_MODE)
        for c in behs:
            behavior_job(model, rd, c)
        save_json({'finished': time.strftime('%Y-%m-%d %H:%M:%S')}, os.path.join(rd, 'done.json'))
    except Exception as e:
        print(f'FAILED: {e}')
        traceback.print_exc()
        with open(os.path.join(OUT_DIR, 'errors.log'), 'a') as f:
            f.write(f'[{time.strftime("%Y-%m-%d %H:%M:%S")}] {name}: {e}\n{traceback.format_exc()}\n')
    finally:
        del model, ctx
        if DEVICE.type == 'cuda':
            torch.cuda.empty_cache()
print(f'\nExecution finished; total notebook time {hours_used():.2f} h.')

## 12. Aggregation: paper tables

| File | Content |
|---|---|
| `T0a_source_properties.csv` | image source properties per class |
| `T0b_metadata_probe.csv` | metadata-only classifier |
| `T0c_duplicate_pairs.csv`, `T0d_duplicates_by_class.csv` | pixel-confirmed duplicates |
| `per_image_properties.csv` | per-image properties (original size, file mode, background, duplicate status) |
| `T1_accuracy.csv` | test accuracy of clean models (all and deduplicated), mean ± SD across seeds |
| `T2_spectral.csv` | spectral estimates and fidelity controls |
| `T3_verdicts_primary.csv` | **primary verdicts**: deduplicated, budget 2×, consensus across seeds |
| `T3_verdicts_dedup_b1.csv`, `T3_verdicts_all_b2.csv` | sensitivity analyses |
| `T3b_effect_sizes.csv`, `T3c_background_arm.csv` | effect sizes per arm and background arm (deduplicated, 2×) |
| `T4_synthetic_detail.csv` | synthetic validation per run × map × budget |
| `T4_synthetic_detection.csv` | sensitivity/specificity of each arm per condition (ground truth = oracle) |
| `T4c_synthetic_effects.csv` | oracle, tumour and background effects per condition × model type |
| `T5_fill_sensitivity.csv` | verdicts under different fills (first run) |
| `T6_behavior.csv` | behaviour on clean, marked and all-marked test sets |
| `T7_tumour_erasure.csv` | correct predictions retained after erasing the tumour (1× = tumour only, 2× = tumour + ring) |
| `T8_fingerprint_probes.csv` | class information outside the lesion |
| `all_audit_summaries.csv` | all statistics (supplementary) |

In [ ]:
# ===================================================================
# 12) AGGREGATION
# ===================================================================
VERDICT_ORDER = ['PATHOLOGY-GROUNDED', 'MIXED', 'OFF-TUMOR', 'REGION-SPECIFIC', 'NON-SPECIFIC', 'INERT',
                 'INCONCLUSIVE', 'INSUFFICIENT']
VERDICT_COLORS = ['#4caf50', '#ffc107', '#e53935', '#8e6cc9', '#ff8a65', '#cfd8dc', '#9e9e9e', '#ffffff']


def short_method(m):
    return m.replace('spectral_subspace', 'spectral\nsubspace')


def mean_sd(v):
    v = np.asarray(v, float)
    v = v[np.isfinite(v)]
    if len(v) == 0:
        return ''
    return f'{v.mean():.4f}' if len(v) == 1 else f'{v.mean():.4f} ± {v.std(ddof=1):.4f}'


def consensus(v):
    vc = pd.Series(v).value_counts()
    return f'{vc.index[0]} ({vc.iloc[0]}/{len(v)})'


acc_rows, spec_rows, summ = [], [], []
for rd in sorted(glob.glob(os.path.join(OUT_DIR, 'runs', '*'))):
    m = RUN_RE.match(os.path.basename(rd))
    if not m:
        continue
    kind = 'clean' if m.group(1) == 'clean' else 'shortcut'
    cond, arch, seed = m.group(2), m.group(3), int(m.group(4))
    if os.path.exists(os.path.join(rd, 'test_metrics.json')):
        t = load_json(os.path.join(rd, 'test_metrics.json'))
        acc_rows.append(dict(kind=kind, cond=cond, arch=arch, seed=seed, acc=t['acc'], bal_acc=t['bal_acc'],
                             macro_f1=t['macro_f1'], acc_dedup=t.get('acc_dedup', np.nan),
                             bal_acc_dedup=t.get('bal_acc_dedup', np.nan),
                             **{f'recall_{k}': v for k, v in t['recall'].items()}))
    if os.path.exists(os.path.join(rd, 'spectral_summary.json')):
        s = load_json(os.path.join(rd, 'spectral_summary.json'))
        spec_rows.append(dict(kind=kind, cond=cond, arch=arch, seed=seed, D=s['spectrum']['D'],
                              K_mp=s['spectrum']['K_mp'], K_mp_asym=s['spectrum'].get('K_mp_asym', np.nan),
                              K_gd=s['spectrum']['K_gd'], K_pa=s['spectrum']['K_pa'],
                              **{k: v for k, v in s['fidelity'].items() if k != 'K'}))
    for f in glob.glob(os.path.join(rd, 'audit_*_summary.csv')):
        mm = re.match(r'audit_(clean|syn-[A-Za-z0-9]+)_(.+)_summary\.csv', os.path.basename(f))
        if not mm:
            continue
        d = pd.read_csv(f)
        d.insert(0, 'fill', mm.group(2)); d.insert(0, 'test_set', mm.group(1))
        d.insert(0, 'seed', seed); d.insert(0, 'arch', arch); d.insert(0, 'cond', cond); d.insert(0, 'kind', kind)
        summ.append(d)

ACC = pd.DataFrame(acc_rows)
SPEC = pd.DataFrame(spec_rows)
SUMM = pd.concat(summ, ignore_index=True) if summ else pd.DataFrame()
if len(SUMM):
    SUMM.to_csv(os.path.join(TD, 'all_audit_summaries.csv'), index=False)

# T1 — accuracy
if len(ACC):
    c = ACC[ACC.kind == 'clean']
    T1 = c.groupby('arch').agg({k: mean_sd for k in ['acc', 'bal_acc', 'macro_f1', 'acc_dedup', 'bal_acc_dedup']})
    T1['n_seeds'] = c.groupby('arch').size()
    T1.to_csv(os.path.join(TD, 'T1_accuracy.csv'))
    print('T1 — Test accuracy of clean models (mean ± SD across seeds)')
    print(T1.to_string(), '\n')

# T2 — spektral
if len(SPEC):
    c = SPEC[SPEC.kind == 'clean']
    cols = [k for k in ['K_mp', 'K_mp_asym', 'K_gd', 'K_pa', 'var_frac_topK', 'weight_energy_topK', 'acc_full',
                        'acc_topK', 'acc_spanW', 'acc_complement', 'acc_randK_mean', 'probe_full',
                        'probe_complement', 'chance'] if k in c]
    T2 = c.groupby('arch')[cols].agg(mean_sd).T
    T2.to_csv(os.path.join(TD, 'T2_spectral.csv'))
    print('T2 — Spectral analysis & fidelity controls (clean models)')
    print(T2.to_string(), '\n')

# T3 — verdicts on real data
if len(SUMM):
    CL = SUMM[(SUMM.kind == 'clean') & (SUMM.test_set == 'clean') & (SUMM.fill == FILL_MODE)]
    for sub, b, fname in [('dedup', PRIMARY_BUDGET, 'T3_verdicts_primary.csv'),
                          ('dedup', BUDGETS[0], f'T3_verdicts_dedup_b{BUDGETS[0]:g}.csv'),
                          ('all', PRIMARY_BUDGET, f'T3_verdicts_all_b{PRIMARY_BUDGET:g}.csv')]:
        c = CL[(CL.subset == sub) & np.isclose(CL.budget, b)]
        if len(c):
            Tv = c.pivot_table(index='cls', columns=['arch', 'method'], values='verdict', aggfunc=consensus).reindex(CLASS_NAMES)
            Tv.to_csv(os.path.join(TD, fname))
    c = CL[(CL.subset == 'dedup') & np.isclose(CL.budget, PRIMARY_BUDGET)]
    if len(c):
        T3 = c.pivot_table(index='cls', columns=['arch', 'method'], values='verdict', aggfunc=consensus).reindex(CLASS_NAMES)
        print(f'T3 — Primary verdicts (clean models, deduplicated, fill={FILL_MODE}, budget {PRIMARY_BUDGET:g}×; consensus across seeds)')
        with pd.option_context('display.width', 250, 'display.max_colwidth', 60):
            print(T3.T.to_string(), '\n')
        T3e = c.groupby(['arch', 'method', 'cls'])[['spec_art_rel', 'r_art', 'spec_tum_rel', 'r_tum', 'art_minus_tum_rel',
                                                    'r_art_tum', 'flip_tum', 'flip_rand', 'frac_tumor']].agg(mean_sd)
        T3e.to_csv(os.path.join(TD, 'T3b_effect_sizes.csv'))
        cb = c[c['method'] == MAPS[0]]
        T3c = cb.groupby(['arch', 'cls']).agg(
            background_reliance=('background_reliance', lambda v: f'{int(to_bool(v).sum())}/{len(v)} seed'),
            bg_rel=('bg_rel', mean_sd), flip_bg=('flip_bg', mean_sd), bg_frac=('bg_frac', mean_sd))
        T3c.to_csv(os.path.join(TD, 'T3c_background_arm.csv'))
        print('T3c — Background arm (clean models, deduplicated)')
        print(T3c.to_string(), '\n')
        keys = list(T3.columns)
        cat = np.full((N_CLASSES, len(keys)), np.nan)
        for j, k in enumerate(keys):
            for i_, cl in enumerate(CLASS_NAMES):
                v = T3.loc[cl, k] if cl in T3.index else np.nan
                if isinstance(v, str):
                    cat[i_, j] = next((n for n, p in enumerate(VERDICT_ORDER) if v.startswith(p)), np.nan)
        fig, ax = plt.subplots(figsize=(max(5, 0.75 * len(keys)), 2.8))
        ax.imshow(cat, cmap=ListedColormap(VERDICT_COLORS), vmin=-0.5, vmax=len(VERDICT_ORDER) - 0.5, aspect='auto')
        for i_ in range(N_CLASSES):
            for j, k in enumerate(keys):
                v = T3.iloc[i_][k]
                if isinstance(v, str):
                    ax.text(j, i_, v.split(' (')[0].replace(' [', '\n[').replace('PATHOLOGY-', 'PATHOLOGY-\n'),
                            ha='center', va='center', fontsize=4)
        ax.set_xticks(range(len(keys)))
        ax.set_xticklabels([f'{a}\n{short_method(m_)}' for a, m_ in keys], fontsize=5, rotation=90)
        ax.set_yticks(range(N_CLASSES))
        ax.set_yticklabels(CLASS_NAMES, fontsize=7)
        ax.set_title(f'Verdicts (clean models, deduplicated test set, {PRIMARY_BUDGET:g}× budget; consensus across seeds)', fontsize=8)
        save_fig(fig, os.path.join(FD, 'verdict_matrix.pdf'), show=True)

# T4 — synthetic validation
if len(SUMM) and RUN_SYNTHETIC:
    s = SUMM[SUMM.test_set.str.startswith('syn-') & (SUMM.subset == 'marker') & (SUMM.label == TARGET_IDX)
             & (SUMM.fill == FILL_MODE)].copy()
    if len(s):
        v = s.verdict.astype(str)
        off_region = (v.str.startswith('OFF-TUMOR') | v.str.startswith('MIXED')
                      | (v.str.startswith('REGION-SPECIFIC') & ~v.str.contains('on-tumor')))
        s['syn_condition'] = s.test_set.str[len('syn-'):]
        s['model'] = np.where(s.kind == 'shortcut', 'shortcut-trained', 'clean (control)')
        s['oracle'] = to_bool(s.oracle_marker_reliance).values
        s['map_detect'] = off_region.values & (s.frac_marker_zone.astype(float) >= 0.5).values
        s['bg_detect'] = to_bool(s.background_marker_reliance).values
        s['tumour_grounded'] = to_bool(s.tumor_grounded).values
        T4 = s[['syn_condition', 'model', 'arch', 'seed', 'method', 'budget', 'n_valid', 'oracle', 'oracle_marker_rel',
                'flip_marker', 'verdict', 'map_detect', 'frac_marker_zone', 'marker_recall', 'bg_detect',
                'bg_marker_rel', 'bg_rel', 'tumour_grounded', 'spec_tum_rel', 'spec_art_rel', 'r_art', 'q_art',
                'art_minus_tum_rel']].sort_values(['syn_condition', 'budget', 'method', 'model', 'seed'])
        T4.to_csv(os.path.join(TD, 'T4_synthetic_detail.csv'), index=False)
        det = []
        for (cnd, meth, b), g in T4.groupby(['syn_condition', 'method', 'budget']):
            pos, neg = g[g.oracle], g[~g.oracle]
            det.append(dict(syn_condition=cnd, arm=f'Arm 1 · {meth}', budget=b, n_pos=len(pos),
                            sensitivity=pos.map_detect.mean() if len(pos) else np.nan, n_neg=len(neg),
                            specificity=(~neg.map_detect).mean() if len(neg) else np.nan))
        g1 = T4[(T4['method'] == MAPS[0]) & np.isclose(T4.budget, PRIMARY_BUDGET)]
        for cnd, g in g1.groupby('syn_condition'):
            pos, neg = g[g.oracle], g[~g.oracle]
            det.append(dict(syn_condition=cnd, arm='Background arm (marker-induced)', budget=np.nan, n_pos=len(pos),
                            sensitivity=pos.bg_detect.mean() if len(pos) else np.nan, n_neg=len(neg),
                            specificity=(~neg.bg_detect).mean() if len(neg) else np.nan))
        T4s = pd.DataFrame(det)
        T4s.to_csv(os.path.join(TD, 'T4_synthetic_detection.csv'), index=False)
        T4c = g1.groupby(['syn_condition', 'model']).agg(
            n_runs=('seed', 'size'), oracle_uses_marker=('oracle', lambda x: f'{int(x.sum())}/{len(x)}'),
            oracle_marker_rel=('oracle_marker_rel', mean_sd), tumour_arm_rel=('spec_tum_rel', mean_sd),
            bg_marker_rel=('bg_marker_rel', mean_sd))
        T4c.to_csv(os.path.join(TD, 'T4c_synthetic_effects.csv'))
        print('T4 — Synthetic shortcut validation (ground truth = oracle)')
        print(T4c.to_string(), '\n')
        with pd.option_context('display.width', 200):
            print(T4s[(T4s.budget.isna()) | np.isclose(T4s.budget.astype(float), PRIMARY_BUDGET)].round(3).to_string(index=False), '\n')
        sp = T4s[(T4s.budget.isna()) | np.isclose(T4s.budget.astype(float), PRIMARY_BUDGET)]
        fig, ax = plt.subplots(1, 2, figsize=(10, 3.0), sharey=True)
        for k, col in enumerate(['sensitivity', 'specificity']):
            p = sp.pivot_table(index='arm', columns='syn_condition', values=col, aggfunc='first')
            p.index = [a.replace('Arm 1 · ', 'Arm 1: ').replace('spectral_subspace', 'spectral subspace') for a in p.index]
            p.plot.bar(ax=ax[k], rot=30, width=0.8, legend=(k == 0))
            ax[k].set_title(col.capitalize() + f' (ground truth = oracle; Arm 1 at {PRIMARY_BUDGET:g}×)', fontsize=8)
            ax[k].set_ylim(0, 1.05); ax[k].set_xlabel('')
            plt.setp(ax[k].get_xticklabels(), fontsize=6, ha='right')
        if ax[0].get_legend():
            ax[0].legend(fontsize=6, title='condition', title_fontsize=6)
        fig.suptitle(f'Synthetic shortcut validation (target class: {SYN_TARGET})', fontsize=9)
        fig.tight_layout(rect=(0, 0, 1, 0.92))
        save_fig(fig, os.path.join(FD, 'synthetic_validation.pdf'), show=True)

# T5 — fill sensitivity
if len(SUMM):
    s = SUMM[(SUMM.kind == 'clean') & (SUMM.arch == FIRST[0]) & (SUMM.seed == FIRST[1])
             & (SUMM.test_set == 'clean') & (SUMM.subset == 'dedup') & np.isclose(SUMM.budget, PRIMARY_BUDGET)]
    if s.fill.nunique() > 1:
        T5 = s.pivot_table(index=['method', 'cls'], columns='fill', values='verdict', aggfunc='first')
        T5.to_csv(os.path.join(TD, 'T5_fill_sensitivity.csv'))
        print('T5 — Sensitivity to the fill method (deduplicated, primary budget)')
        with pd.option_context('display.width', 250, 'display.max_colwidth', 45):
            print(T5.to_string(), '\n')

# T6 — behaviour
beh_rows = []
for f in glob.glob(os.path.join(OUT_DIR, 'runs', '*', 'behavior_*.json')):
    m = RUN_RE.match(os.path.basename(os.path.dirname(f)))
    if not m:
        continue
    bcond = os.path.basename(f)[len('behavior_'):-len('.json')]
    for ts, v in load_json(f).items():
        beh_rows.append(dict(kind='clean' if m.group(1) == 'clean' else 'shortcut', syn_condition=bcond,
                             arch=m.group(3), seed=int(m.group(4)), test=ts, acc=v['acc'],
                             **{f'recall_{k}': x for k, x in v['recall'].items()},
                             **{f'pred_as_target_{k}': x for k, x in v['pred_as_target'].items()}))
if beh_rows:
    T6 = pd.DataFrame(beh_rows).sort_values(['syn_condition', 'kind', 'seed', 'test'])
    T6.to_csv(os.path.join(TD, 'T6_behavior.csv'), index=False)
    print('T6 — Behaviour (target recall without markers, and target predictions when every image is marked)')
    _t6 = T6[T6['test'].isin(['clean', 'all_marked'])]
    print(_t6.groupby(['syn_condition', 'kind', 'test'])[[f'recall_{SYN_TARGET}'] + [f'pred_as_target_{c}' for c in CLASS_NAMES if c != SYN_TARGET]]
          .agg(mean_sd).to_string(), '\n')

# T7 — correct predictions retained after erasing the tumour (1× = tumour only; 2× = tumour + equal-area ring)
t7 = []
for rd in sorted(glob.glob(os.path.join(OUT_DIR, 'runs', 'clean_*'))):
    m = RUN_RE.match(os.path.basename(rd))
    if not m:
        continue
    for f in glob.glob(os.path.join(rd, 'audit_clean_*_images.csv')):
        fill = re.match(r'audit_clean_(.+)_images\.csv', os.path.basename(f)).group(1)
        d = pd.read_csv(f, usecols=['method', 'budget', 'label', 'correct0', 'has_mask', 'dup', 'flip_tum'])
        d = d[(d['method'] == MAPS[0]) & to_bool(d.correct0) & to_bool(d.has_mask) & ~to_bool(d.dup)]
        for (b, c), dc in d.groupby(['budget', 'label']):
            if int(c) == 0 or len(dc) == 0:
                continue
            t7.append(dict(arch=m.group(3), seed=int(m.group(4)), fill=fill, budget=float(b), cls=CLASS_NAMES[int(c)],
                           n=len(dc), retained=float(1 - dc.flip_tum.astype(float).mean())))
if t7:
    T7raw = pd.DataFrame(t7)
    T7 = T7raw.groupby(['fill', 'budget', 'arch', 'cls']).retained.agg(mean_sd).unstack('cls')
    T7.to_csv(os.path.join(TD, 'T7_tumour_erasure.csv'))
    print('T7 — Correct predictions retained after erasing the tumour (deduplicated; 1× = tumour only, 2× = tumour + equal-area ring)')
    print(T7.to_string(), '\n')
    pr = T7raw[(T7raw['fill'] == FILL_MODE) & np.isclose(T7raw.budget, PRIMARY_BUDGET)] \
        .groupby(['cls', 'arch']).retained.mean().unstack('arch')
    if len(pr):
        fig, ax = plt.subplots(figsize=(5.5, 2.8))
        pr.reindex([c for c in CLASS_NAMES if c in pr.index]).plot.bar(ax=ax, rot=0, width=0.75)
        ax.set_ylim(0, 1.05)
        ax.set_ylabel('Correct predictions retained')
        ax.set_xlabel('')
        ax.set_title(f'Predictions retained after erasing the tumour ({PRIMARY_BUDGET:g}× area, fill={FILL_MODE})', fontsize=8)
        ax.legend(fontsize=6)
        save_fig(fig, os.path.join(FD, 'tumour_erasure.pdf'), show=True)

# T8 — fingerprint probes
if len(T8):
    print('T8 — Class information outside the lesion (deduplicated)')
    print(T8[T8.subset == 'dedup'].round(3).to_string(index=False))

if os.path.exists(os.path.join(OUT_DIR, 'errors.log')):
    print('\nWARNING: some runs failed or were skipped — see', os.path.join(OUT_DIR, 'errors.log'))

In [ ]:
# ===================================================================
# 13) RESULTS ARCHIVE (without checkpoints and cache)
# ===================================================================
zip_path = OUT_DIR.rstrip('/') + '_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(OUT_DIR):
        if os.sep + 'cache' in root:
            continue
        for f in files:
            if f.endswith('.pt'):
                continue
            p = os.path.join(root, f)
            zf.write(p, os.path.relpath(p, os.path.dirname(OUT_DIR)))
print(f'Archive: {zip_path} ({os.path.getsize(zip_path) / 1e6:.1f} MB)')